# Last Modified: 2026-04-23 15:00:00

---

# LFP Battery SOH Preprocessing Pipeline (Universal Version)

This notebook implements a physically-consistent preprocessing framework for multi-source battery datasets:
1. **Phase 0**: Data Loading (HUST, MIT-Stanford, etc.)
2. **Phase 0-1**: MIT Data Adapter (Port to HUST DataFrame Format)
3. **Phase 0.5**: Dataset-agnostic Configuration & Helpers
4. **Phase 1**: Global Physical Cleaning (Savitzky-Golay Smoothing & Outlier Removal)
5. **Phase 2**: Scenario-based Slicing Definition
6. **Phase 2.1**: **Global Slicing Loop (Generate & Cache Raw Segment Pool with RUL/Cap labels)**
7. **Phase 3**: 45D Adaptive HI Extraction Logic (Standardized V/I/T/Q features + Physical Features)
8. **Phase 4**: **Feature Extraction & Batch Scaling**
9. **Phase 5**: Global Tensor Saving (Optimized pkl tensors on D: drive)
10. **Phase 6**: Dataset-wide Visualization & PCA Analysis

In [1]:
import numpy as np
import pandas as pd
import gc
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
from tqdm.notebook import tqdm
import os
import sys
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.stats import skew, kurtosis
from scipy.signal import savgol_filter
from collections import defaultdict
import random
from tqdm.auto import tqdm
from collections import defaultdict
from scipy.signal import savgol_filter
import pdb

from datetime import date

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    
from src.hyperparams import *
from src.data.loaders import load_hust, load_mit, calc_Q

# 오늘 날짜 객체 생성 
today = date.today()

figure_path = Path(f"../../outputs/figures/{HYPERPARAMS['major_version']}/{today.strftime('%m%d')}")
figure_path.mkdir(parents=True, exist_ok=True)

DATASET_TYPE = "hust" # Switch to "hust" or "mit" as needed

# Define standardized column constants for internal logic
COL_V = "Voltage (V)"
COL_I = "Current (A)"
COL_T = "Temperature (°C)"
COL_TIME = "Time (s)"

do_remove = True  # removal_index.csv에 있는 사이클을 제거할지 여부
cycle_step = 1  # 몇 사이클마다 데이터를 사용할지 (예: 5면 0,5,10,...)
sliding_step= 0.1  # SOC 시나리오 내에서 슬라이딩 윈도우 간격

hust_cell_ids = [
    '1-1', '1-2', '1-3', '1-4', '1-5', '1-6', '1-7', '1-8', 
    '2-2', '2-3', '2-4', '2-5', '2-6', '2-7', '2-8', 
    '3-1', '3-2', '3-3', '3-4', '3-5', '3-6', '3-7', '3-8', 
    '4-1', '4-2', '4-3', '4-4', '4-5', '4-6', '4-7', '4-8', 
    '5-1', '5-2', '5-3', '5-4', '5-5', '5-6', '5-7', 
    '6-1', '6-2', '6-3', '6-4', '6-5', '6-6', '6-8', 
    '7-1', '7-2', '7-3', '7-4', '7-5', '7-6', '7-7', '7-8', 
    '8-1', '8-2', '8-3', '8-4', '8-5', '8-6', '8-7', '8-8', 
    '9-1', '9-2', '9-3', '9-4', '9-5', '9-6', '9-7', '9-8', 
    '10-1', '10-2', '10-3', '10-4', '10-5', '10-6', '10-7', '10-8'
]

mit_cell_ids = [f"{b}c{i}" for b in ["b1", "b2", "b3"] for i in range(46)] #b1c0~b1c45, b2c0~b2c45, b3c0~b3c45

## Phase 0: Data Loading

In [2]:
# --- [1. Pipeline Control] ---
# 1: Physical Cleaning, 2: Slicing, 3: Feature Extraction, 4: Final Tensor Saving
START_PHASE = 1
FORCE_REGEN = False 

PROJECT_ROOT = Path.cwd().parent.parent
SCENARIO_ANALYSIS_FIGURE_DIR = (
    PROJECT_ROOT 
    / "outputs" 
    / "figures" 
    / "scenario_analysis" 
    / f"case_{HYPERPARAMS['major_version']}" 
    / DATASET_TYPE
)

# --- [2. Global Path Management] ---
PROCESSED_DATA_ROOT = Path(f"D:/chanminLee/data_store/LFP_SOH_estimation/case_{HYPERPARAMS['major_version']}")
FIGURE_ROOT = PROJECT_ROOT / "outputs" / "figures" / str(HYPERPARAMS['major_version']) / date.today().strftime('%m%d')

# Phase 1: Physical Cleaning
CLEAN_CACHE_DIR = PROCESSED_DATA_ROOT / f"{DATASET_TYPE}_clean_chunks"
CLEAN_REPORT_PATH = PROJECT_ROOT / "outputs" / f"{DATASET_TYPE}_cleaning_report.csv"

# Phase 2: Scenario-based Slicing
CACHE_DIR = PROCESSED_DATA_ROOT / f"{DATASET_TYPE}_sliced_chunks"

# Phase 3: Feature Extraction & Scaling
FEATURE_BATCH_DIR = PROCESSED_DATA_ROOT / f"{DATASET_TYPE}_feature_batches"
SCALER_CACHE_PATH = PROCESSED_DATA_ROOT / f"{DATASET_TYPE}_scaler.pkl"

# Phase 4: Final Tensor Saving
FINAL_TENSOR_PATH = PROCESSED_DATA_ROOT / f"{DATASET_TYPE}_optimized_tensors.pkl"

# Ensure all directories exist
for p in [SCENARIO_ANALYSIS_FIGURE_DIR, PROCESSED_DATA_ROOT, FIGURE_ROOT, CLEAN_CACHE_DIR, CACHE_DIR, FEATURE_BATCH_DIR, PROCESSED_DATA_ROOT / DATASET_TYPE]:
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
if START_PHASE <= 1:
    print(f"Starting Phase 1: Physical Cleaning for {DATASET_TYPE.upper()} dataset...")
    if DATASET_TYPE == "hust":
        print("Loading HUST dataset...")
        all_cells = load_hust(PROJECT_ROOT / "HUST_data" / "data")
    elif DATASET_TYPE == "mit":
        print("Loadin`g MIT-Stanford dataset...")
        all_cells = load_mit(PROJECT_ROOT / "MIT_data")
    print(f"Loaded {len(all_cells)} cells for {DATASET_TYPE.upper()} dataset.")


Starting Phase 1: Physical Cleaning for HUST dataset...
Loading HUST dataset...


HUST Cells:   0%|          | 0/77 [00:00<?, ?it/s]

## Phase 0-1: Data Adapter

In [ ]:
if START_PHASE <= 1:
    if DATASET_TYPE == "mit":
        print("Porting MIT data to Unified DataFrame format...")
        for cid, cell in tqdm(all_cells.items(), desc="Converting MIT"):
            cell.data = {}
            for cyc_key, cyc_dict in cell.cycles.items():
                if int(cyc_key) > 0:
                    q_total = np.where(cyc_dict['I'] >= 0, cyc_dict['Qc'], cyc_dict['Qd'])
                    cell.data[int(cyc_key)] = pd.DataFrame({
                        COL_TIME: cyc_dict['t'], COL_V: cyc_dict['V'],
                        COL_I: cyc_dict['I'], COL_T: cyc_dict['T'], 'capacity': q_total
                    })
    elif DATASET_TYPE == "hust":
        print("Standardizing HUST data...")
        for cid, cell in tqdm(all_cells.items(), desc="Standardizing HUST"):
            for cyc in list(cell.data.keys()):
                df = cell.data[cyc].copy()
                if 'Capacity (mAh)' in df.columns: df['capacity'] = df['Capacity (mAh)'] / 1000.0
                if 'Current (mA)' in df.columns: df[COL_I] = df['Current (mA)'] / 1000.0
                rename_dict = {'Voltage (V)': COL_V, 'Temperature (degC)': COL_T, 'Time (s)': COL_TIME}
                cell.data[cyc] = df.rename(columns={k: v for k, v in rename_dict.items() if k in df.columns})

Porting MIT data to Unified DataFrame format...


Converting MIT:   0%|          | 0/123 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
if START_PHASE <= 1:
    # --- Unified Data Structure Audit ---
    print(f"=== 📊 {DATASET_TYPE.upper()} Unified Dataset Audit ===")
    total_cells = len(all_cells)
    cycle_counts = [len(cell.data) for cell in all_cells.values()]
    print(f"1. Total Cells loaded: {total_cells}")
    print(f"2. Cycle Stats per Cell: Min={np.min(cycle_counts)}, Max={np.max(cycle_counts)}, Avg={np.mean(cycle_counts):.1f}")

    # Extract sample data for schema inspection
    sample_cid = list(all_cells.keys())[0]
    sample_cyc = list(all_cells[sample_cid].data.keys())[0]
    sample_df = all_cells[sample_cid].data[sample_cyc]

    print(f"\n3. Unified DataFrame Schema (Cell {sample_cid}, Cyc {sample_cyc}):")
    print(f"   - Columns: {sample_df.columns.tolist()}")
    print(f"   - Shape: {sample_df.shape}")
    print(f"   - Data Types:\n{sample_df.dtypes}")

    # Check for essential features for preprocessing
    has_time = COL_TIME in sample_df.columns
    has_v = COL_V in sample_df.columns
    has_i = COL_I in sample_df.columns

    print(f"\n4. Preprocessing Readiness Check:")
    print(f"   - Time info available: {has_time}")
    print(f"   - Voltage info available: {has_v}")
    print(f"   - Current info available: {has_i}")

    # sample data preview
    print(f"\n5. Sample Data Preview (Cell {sample_cid}, Cyc {sample_cyc}):")
    print(sample_df.head())
    print(sample_df.tail(), '\n')
    print(sample_df.describe())

    if DATASET_TYPE == 'mit':
        print("   - [MIT Note] Current converted to Ampere (A) from loader.")
    elif DATASET_TYPE == 'hust':
        print("   - [HUST Note] Voltage smoothing and Current unit conversion (mA->A) pending in Phase 1.")

    print("===========================================")

In [ ]:
os.makedirs(PROCESSED_DATA_ROOT / f"{DATASET_TYPE}", exist_ok=True)

if START_PHASE <= 1:
    if DATASET_TYPE == "mit":
        # sample_cell = next(iter(all_cells.values()))
        # Basic EDA: Check column availability and data quality
        print("\n=== Basic EDA ===")
        sample_cid = list(all_cells.keys())[0]
        sample_cell = all_cells[sample_cid]
        print(f"Sample Cell ID: {sample_cid}")
        print(f"Available columns: {sample_cell.__dict__.keys()}")
        print(f"Available summary columns: {sample_cell.summary.keys()}")
        print(f"Available cycles columns: {sample_cell.cycles.keys()}")
        print(f"Available cycle 1 columns: {sample_cell.cycles['1'].keys()}")
        print(f"summary data each columns length: { {k: len(v) for k, v in sample_cell.summary.items()} }")
        print(f":{sample_cell.summary}")

        cycle_dict = sample_cell.cycles['1']
        cycle_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in cycle_dict.items()]))

        pd.DataFrame(sample_cell.summary).to_csv(PROCESSED_DATA_ROOT / f"mit/{sample_cid}_summary.csv", index=False) # Save raw data for inspection   
        pd.DataFrame(cycle_df).to_csv(PROCESSED_DATA_ROOT / f"mit/{sample_cid}_cycles.csv", index=False) # Save raw data for inspection   

    elif "hust":
        # sample_cell = next(iter(all_cells.values()))
        # Basic EDA: Check column availability and data quality
        print("\n=== Basic EDA ===")
        sample_cid = list(all_cells.keys())[0]
        sample_cell = all_cells[sample_cid]
        print(f"Sample Cell ID: {sample_cid}")
        print(f"Available columns: {sample_cell.__dict__.keys()}")
        print(f'dq columns: {sample_cell.dq}') # mAh 단위 
        print(f'data columns: {sample_cell.data[1]}') 
        sample_cell.data[1].to_csv(PROCESSED_DATA_ROOT / f"hust/{sample_cid}_raw.csv", index=False) # Save raw data for inspection

## Phase 1: Global Physical Cleaning

In [ ]:
print("\n=== Basic EDA & Sample Export ===")
sample_cid = list(all_cells.keys())[0]
sample_cell = all_cells[sample_cid]
print(f"Sample Cell ID: {sample_cid}")

if DATASET_TYPE == "mit":
    cycle_dict = sample_cell.cycles['1']
    cycle_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in cycle_dict.items()]))
    pd.DataFrame(sample_cell.summary).to_csv(PROCESSED_DATA_ROOT / f"{DATASET_TYPE}/{sample_cid}_summary.csv", index=False)
    pd.DataFrame(cycle_df).to_csv(PROCESSED_DATA_ROOT / f"{DATASET_TYPE}/{sample_cid}_cycles.csv", index=False)
elif DATASET_TYPE == "hust":
    sample_cell.data[1].to_csv(PROCESSED_DATA_ROOT / f"{DATASET_TYPE}/{sample_cid}_raw.csv", index=False)

def get_cell_labels(cell, cyc, dataset_type):
    """Extract RUL and SOH (Capacity) labels in a dataset-agnostic way."""
    if dataset_type == "hust":
        return cell.rul[cyc], cell.dq[cyc]/1000.0  # Convert mAh to Ah for consistency
    elif dataset_type == "mit":
        rul = (cell.cycle_life - cyc) if cell.cycle_life else 0
        cap = cell.summary['QD'][cyc-1] if (cyc-1) < len(cell.summary['QD']) else 0.0
        return float(rul), float(cap)
    return 0.0, 0.0

def get_batch_id(cid, dataset_type):
    """Standardize batch identification for grouping chunks."""
    if dataset_type == "hust":
        return str(cid).split("-")[0]
    elif dataset_type == "mit":
        return str(cid).split("c")[0]
    return "batch_0"

In [ ]:
import pandas as pd
import numpy as np
import os
import gc
from pathlib import Path
from tqdm.auto import tqdm

# 이상치가 제거되지 않은 all_cells를 통해 시나리오별로 데이터를 슬라이스하고 통계량을 계산하는 함수
def get_scenario_stats(df, cid, cyc, mode, scenario, start_p, end_p, target_q, col_v, col_i, col_t, col_temp):
    """
    Slices the cycle dataframe based on mode and SOC scenario, then calculates statistics.
    """
    # 1. Mode Filtering
    if mode == "C":
        mode_df = df[df[col_i] > 0.01].copy()
    else:
        mode_df = df[df[col_i] < -0.01].copy()

    if mode_df.empty:
        return None

    # 2. Capacity Accumulation
    if 'capacity' in mode_df.columns:
        q_acc = (mode_df['capacity'] - mode_df['capacity'].iloc[0]).abs()
    else:
        dt = mode_df[col_t].diff().fillna(0)
        dq = (mode_df[col_i].abs() * dt) / 3600.0
        q_acc = dq.cumsum()

    # 3. Slicing
    q_ratio = q_acc / (target_q + 1e-9)
    mask = (q_ratio >= start_p) & (q_ratio <= end_p)
    sliced = mode_df[mask]

    if sliced.empty:
        return None

    # 4. Statistics Calculation
    stats = {
        'cell_id': cid,
        'cycle': cyc,
        'count': len(sliced),
        'time': sliced[col_t].mean()
    }

    for feat_name, col_name in [('V', col_v), ('I', col_i), ('T', col_temp)]:
        if col_name in sliced.columns:
            series = sliced[col_name]
            stats[f'{feat_name}_mean'] = series.mean()
            stats[f'{feat_name}_std'] = series.std()
            stats[f'{feat_name}_var'] = series.var()
            stats[f'{feat_name}_min'] = series.min()
            stats[f'{feat_name}_max'] = series.max()
        else:
            stats[f'{feat_name}_mean'] = np.nan
            stats[f'{feat_name}_std'] = np.nan
            stats[f'{feat_name}_var'] = np.nan
            stats[f'{feat_name}_min'] = np.nan
            stats[f'{feat_name}_max'] = np.nan

    return stats


def process_scenarios_from_memory(dataset_type, all_cells_dict, output_root, get_cell_labels_func):
    """
    Data Adapting(Phase 0.1) 직후 메모리의 all_cells_dict를 직접 순회하며
    6개 시나리오 통계 CSV를 생성합니다. (Cache 로드 불필요)
    """
    scenarios = {
        'charge-high': ('C', 0.0, 0.3),
        'charge-mid': ('C', 0.3, 0.7),
        'charge-low': ('C', 0.7, 1.0),
        'discharge-high': ('D', 0.0, 0.3),
        'discharge-mid': ('D', 0.3, 0.7),
        'discharge-low': ('D', 0.7, 1.0)
    }

    # 1. Initialize CSV files with headers
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    csv_paths = {}
    cols = [
        'cell_id', 'cycle', 'count', 'time',
        'V_mean', 'V_std', 'V_var', 'V_min', 'V_max',
        'I_mean', 'I_std', 'I_var', 'I_min', 'I_max',
        'T_mean', 'T_std', 'T_var', 'T_min', 'T_max'
    ]

    for name in scenarios.keys():
        path = output_root / f"{dataset_type}_{name}_summary.csv"
        pd.DataFrame(columns=cols).to_csv(path, index=False)
        csv_paths[name] = path

    # 2. Column name mapping (Data Adapting 직후 표준화된 컬럼명 가정)
    col_v, col_i, col_t = COL_V, COL_I, COL_TIME
    col_temp = COL_T if dataset_type == "mit" else 'Temp'  # HUST에 온도가 없다면 임의 지정

    # 3. 메모리의 all_cells 직접 순회
    for cid, cell_obj in tqdm(all_cells_dict.items(), desc=f"Processing Cells ({dataset_type.upper()})"):
        cell_results = {name: [] for name in scenarios.keys()}

        # cell_obj.data 에는 사이클별 DataFrame이 저장되어 있음
        if not hasattr(cell_obj, 'data') or not cell_obj.data:
            print(f"Warning: Cell {cid} has no data.")
            continue

        cycles = sorted(cell_obj.data.keys(), key=lambda x: int(x))

        for cyc in tqdm(cycles, desc=f"  Cycles of {cid}", leave=False):
            df_cyc = cell_obj.data[cyc]

            # 용량 라벨 가져오기
            _, cap_label = get_cell_labels_func(cell_obj, cyc, dataset_type)

            if cap_label <= 0:
                continue

            # 각 시나리오별 통계량 추출
            for scen_name, (mode, start_p, end_p) in scenarios.items():
                res = get_scenario_stats(
                    df_cyc, cid, cyc, mode, scen_name,
                    start_p, end_p, cap_label,
                    col_v, col_i, col_t, col_temp
                )
                if res:
                    cell_results[scen_name].append(res)

        # 한 셀의 처리가 끝나면 CSV에 바로 Append 하여 메모리 누수 방지
        for scen_name, results in cell_results.items():
            if results:
                pd.DataFrame(results).to_csv(
                    csv_paths[scen_name],
                    mode='a',
                    header=False,
                    index=False
                )

        del cell_results
        gc.collect()

    print(f">> Finished generating raw scenario CSVs for {dataset_type.upper()}")

In [ ]:
if START_PHASE <= 1:
    # 1. 결과 저장 경로 설정
    output_dir = PROCESSED_DATA_ROOT / "scenario_summaries"
    output_dir.mkdir(exist_ok=True)
    process_scenarios_from_memory(DATASET_TYPE, all_cells, output_dir, get_cell_labels)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# =============================================================================
# [Helper Functions] 공통 재사용 로직
# =============================================================================
def load_scenario_data(summary_root, dataset_type, scenario_name):
    """지정된 시나리오의 요약 CSV 파일을 로드합니다."""
    path = Path(summary_root) / f"{dataset_type}_{scenario_name}_summary.csv"
    if not path.exists():
        print(f"Warning: File not found: {path}")
        return None
    return pd.read_csv(path)

def plot_feature_trends(df, scenario_name, features_to_plot, cell_ids=None, save_dir=None):
    if df is None or df.empty:
        return

    if cell_ids:
        plot_df = df[df["cell_id"].isin(cell_ids)].copy()
    else:
        unique_cells = df["cell_id"].unique()
        plot_df = df[df["cell_id"].isin(unique_cells[:5])].copy()

    num_features = len(features_to_plot)
    fig, axes = plt.subplots(num_features, 1, figsize=(12, 4 * num_features), sharex=True)
    if num_features == 1:
        axes = [axes]

    for i, feat in enumerate(features_to_plot):
        if feat not in plot_df.columns:
            continue

        sns.lineplot(
            data=plot_df,
            x="cycle",
            y=feat,
            hue="cell_id",
            ax=axes[i],
            marker="o",
            alpha=0.7
        )

        axes[i].set_title(f"[{scenario_name}] Trend of {feat}")
        axes[i].set_ylabel(feat)
        axes[i].grid(True, linestyle="--", alpha=0.6)
        axes[i].legend(bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.xlabel("Cycle Number")
    plt.tight_layout()

    if save_dir:
        save_path = Path(save_dir) / f"{scenario_name}_trends.png"
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        plt.close(fig)

def plot_pareto_distribution(df, scenario_name, features_to_plot, save_dir=None):
    if df is None or df.empty:
        return

    num_features = len(features_to_plot)
    fig, axes = plt.subplots(num_features, 1, figsize=(10, 4 * num_features))

    if num_features == 1:
        axes = [axes]

    for i, feat in enumerate(features_to_plot):
        if feat not in df.columns:
            continue

        data = df[feat].dropna().sort_values()
        if len(data) == 0:
            continue

        y_vals = np.arange(1, len(data) + 1) / len(data)

        axes[i].plot(
            data,
            y_vals,
            marker=".",
            linestyle="none",
            alpha=0.5
        )

        axes[i].set_title(f"[{scenario_name}] Pareto (CDF) of {feat}")
        axes[i].set_xlabel(feat)
        axes[i].set_ylabel("Cumulative Proportion")
        axes[i].grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()

    if save_dir:
        save_path = Path(save_dir) / f"{scenario_name}_pareto.png"
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        plt.close(fig)

# =============================================================================
# [Function 1] 이상치 탐지 및 Removal Index 생성
# =============================================================================
def generate_removal_index(dataset_type, summary_csv_dir, project_root, major_version=1):
    """
    모든 시나리오 CSV를 읽어 타겟 피처의 극단적 이상치를 탐지하고
    프로젝트 루트에 removal_index.csv를 생성합니다.
    """
    print(f"\n[Phase 0.1 Post-task] Generating Removal Index for {dataset_type.upper()}...")

    SUMMARY_ROOT = Path(summary_csv_dir)
    REMOVAL_INDEX_PATH = Path(project_root) / f"case{major_version}_{dataset_type}_removal_index.csv"

    scenarios = [
        "charge-high",
        "charge-mid",
        "charge-low",
        "discharge-high",
        "discharge-mid",
        "discharge-low"
    ]

    target_features = ["V_std", "count", "I_std", "T_std", "V_min", "V_max"]
    all_removal_indices = []

    for scenario in scenarios:
        df = load_scenario_data(SUMMARY_ROOT, dataset_type, scenario)

        if df is None or df.empty:
            continue

        for feat in target_features:
            if feat not in df.columns:
                continue

            feature_values = df[feat].dropna().sort_values()
            n = len(feature_values)

            if n > 0:
                lower_bound = np.percentile(feature_values, 0.1)
                upper_bound = np.percentile(feature_values, 99.9)

                outliers = df[
                    (df[feat] < lower_bound) |
                    (df[feat] > upper_bound)
                ].copy()

                if not outliers.empty:
                    outliers["scenario"] = scenario
                    outliers["feature"] = feat

                    all_removal_indices.append(
                        outliers[
                            ["cell_id", "cycle", "scenario", "feature", feat]
                        ]
                    )

    if all_removal_indices:
        final_removal_df = pd.concat(all_removal_indices, ignore_index=True)

        final_removal_df = final_removal_df.drop_duplicates(
            subset=["cell_id", "cycle"]
        )

        final_removal_df.to_csv(
            REMOVAL_INDEX_PATH,
            index=False
        )

        print(
            f">> Saved {len(final_removal_df)} removal target cycles "
            f"to {REMOVAL_INDEX_PATH}"
        )

    else:
        pd.DataFrame(
            columns=["cell_id", "cycle", "scenario", "feature"]
        ).to_csv(
            REMOVAL_INDEX_PATH,
            index=False
        )

        print(
            f">> No outliers detected. "
            f"Empty index saved to {REMOVAL_INDEX_PATH}"
        )

# =============================================================================
# [Function 2] Removal Index 적용 및 데이터 시각화
# =============================================================================
def visualize_cleaned_scenarios(
    dataset_type,
    summary_csv_dir,
    project_root,
    figure_out_dir,
    major_version=1
):
    """
    생성된 removal_index.csv를 읽어 불량 사이클을 필터링한 후,
    정제된 데이터를 바탕으로 트렌드 및 파레토 플롯을 그립니다.
    """
    print(
        f"\n[Phase 0.1 Post-task] "
        f"Visualizing Cleaned Data for {dataset_type.upper()}..."
    )

    SUMMARY_ROOT = Path(summary_csv_dir)

    FIGURE_DIR = Path(figure_out_dir)
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

    REMOVAL_INDEX_PATH = (
        Path(project_root)
        / f"case{major_version}_{dataset_type}_removal_index.csv"
    )

    scenarios = [
        "charge-high",
        "charge-mid",
        "charge-low",
        "discharge-high",
        "discharge-mid",
        "discharge-low"
    ]

    target_features = [
        "V_std",
        "count",
        "I_std",
        "T_std",
        "V_min",
        "V_max"
    ]

    unique_removals = pd.DataFrame(columns=["cell_id", "cycle"])

    if REMOVAL_INDEX_PATH.exists():
        removal_df = pd.read_csv(REMOVAL_INDEX_PATH)

        if not removal_df.empty:
            unique_removals = removal_df[
                ["cell_id", "cycle"]
            ].drop_duplicates()

    for scenario in scenarios:
        df = load_scenario_data(
            SUMMARY_ROOT,
            dataset_type,
            scenario
        )

        if df is None or df.empty:
            continue

        original_len = len(df)

        if not unique_removals.empty:
            df = df.merge(
                unique_removals,
                on=["cell_id", "cycle"],
                how="left",
                indicator=True
            )

            df = (
                df[df["_merge"] == "left_only"]
                .drop(columns=["_merge"])
            )

        print(
            f"  - [{scenario}] Removed "
            f"{original_len - len(df)} bad cycles. "
            f"Plotting remaining {len(df)} cycles."
        )
        
        plot_feature_trends(
            df,
            scenario,
            target_features,
            save_dir=FIGURE_DIR
        )

        plot_pareto_distribution(
            df,
            scenario,
            target_features,
            save_dir=FIGURE_DIR
        )

    print(
        f">> Visualization complete. "
        f"Check '{FIGURE_DIR}'"
    )
    
if START_PHASE <= 1:
    PROJECT_ROOT = "../../"

    if do_remove is True:
        generate_removal_index(
            DATASET_TYPE,
            output_dir,
            PROJECT_ROOT,
            major_version=HYPERPARAMS['major_version']
        )


    visualize_cleaned_scenarios(
        DATASET_TYPE,
        output_dir,
        PROJECT_ROOT,
        SCENARIO_ANALYSIS_FIGURE_DIR,
        major_version=HYPERPARAMS['major_version']
    )

## phase 1.1 cleaning

In [ ]:
# 0. Global Removal 대상 로드
REMOVAL_PATH = f"../../case{HYPERPARAMS['major_version']}_{DATASET_TYPE}_removal_index.csv"
remove_set = set()

if os.path.exists(REMOVAL_PATH):
    ref_df = pd.read_csv(REMOVAL_PATH)
    remove_set = set(zip(ref_df['cell_id'], ref_df['cycle'].astype(int)))
    print(f">> Loaded {len(remove_set)} cycles to exclude from {REMOVAL_PATH}")
else:
    print(f">> No removal index file found at {REMOVAL_PATH}. Proceeding without global removals.")


def clean_physical_data(df, cid, cyc, cleaning_reports):
    """
    Z-score 기반 포인트 제거 로직이 삭제된 버전.
    단위 표준화, 용량 계산, 결측치 보간 및 smoothing만 수행합니다.
    """
    clean_df = df.copy()

    # 1. 단위 표준화 (mA -> A)
    if "Current (mA)" in clean_df.columns:
        clean_df[COL_I] = clean_df["Current (mA)"] / 1000.0
    elif clean_df[COL_I].abs().mean() > 50:
        clean_df[COL_I] = clean_df[COL_I] / 1000.0

    # 2. 용량(Capacity) 계산 (없을 경우)
    if 'capacity' not in clean_df.columns:
        q_c = calc_Q(
            clean_df[COL_I].values,
            clean_df[COL_TIME].values,
            is_charge=True
        )
        q_d = calc_Q(
            clean_df[COL_I].values,
            clean_df[COL_TIME].values,
            is_charge=False
        )
        clean_df['capacity'] = np.where(
            clean_df[COL_I] >= 0,
            q_c,
            q_d
        )
        
    # 3. 전류가 0인 행 제거 (휴지기, 0.01미만의 cv 구간 전류도 있으므로 0만 제거) 및 리포트 작성
    zero_current_mask = clean_df[COL_I].abs() = 0
    if zero_current_mask.any():
        clean_df = clean_df[~zero_current_mask].reset_index(drop=True)
        cleaning_reports.append({
            "cell_id": cid,
            "cycle": cyc,
            "feature": "Current (A)",
            "outlier_removed": zero_current_mask.sum(),
            "interpolated_count": 0,
            "outlier_details_(idx,val)": str(list(zip(clean_df.index[zero_current_mask], clean_df.loc[zero_current_mask, COL_I]))),
            "nan_indices": ""
        })

    # 4. 결측치(NaN) 체크 및 리포트 작성 (Z-score 로직 삭제됨)
    num_cols = clean_df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        nan_mask = clean_df[col].isna()

        if nan_mask.any():
            cleaning_reports.append({
                "cell_id": cid,
                "cycle": cyc,
                "feature": col,
                "outlier_removed": 0,  # Z-score 제거 로직이 없으므로 항상 0
                "interpolated_count": nan_mask.sum(),
                "outlier_details_(idx,val)": "",
                "nan_indices": str(clean_df.index[nan_mask].tolist())
            })

    return clean_df


# --- Phase 1 실행부 ---
if START_PHASE <= 1 or not any(f.name.startswith("batch_clean_") for f in CLEAN_CACHE_DIR.glob("batch_clean_*")):
    print(f"[Phase 1] Starting Physical Cleaning + Global Removal for {DATASET_TYPE}...")
    batch_groups = defaultdict(dict)

    for cid, cell in all_cells.items():
        b_id = get_batch_id(cid, DATASET_TYPE)
        batch_groups[b_id][cid] = cell

    all_cleaning_reports = []

    for batch_id in sorted(batch_groups.keys()):
        batch_cache_path = CLEAN_CACHE_DIR / f"batch_clean_{batch_id}.pkl"

        # FORCE_REGEN이 False이고 캐시가 있으면 로드만 하고 스킵
        if batch_cache_path.exists() and not FORCE_REGEN:
            report_part_path = CLEAN_CACHE_DIR / f"batch_clean_{batch_id}_report.pkl"

            if report_part_path.exists():
                with open(report_part_path, "rb") as f:
                    all_cleaning_reports.extend(pickle.load(f))
            continue

        batch_cells = batch_groups[batch_id]
        batch_cleaned = {}
        batch_reports = []

        for cid, cell in batch_cells.items():
            batch_cleaned[cid] = {}
            for cyc in tqdm(
                list(cell.data.keys()),
                desc=f"  Cleaning Cell {cid}",
                leave=False
            ):

                # do_remove가 true이고 removal_index.csv에 있는 사이클이면 통째로 스킵
                if do_remove is not False and (cid, int(cyc)) in remove_set:
                    print(f"    >> Skipping Cell {cid} Cycle {cyc} due to global removal index.")
                    batch_reports.append({
                        "cell_id": cid,
                        "cycle": cyc,
                        "feature": "GLOBAL_REMOVAL",
                        "outlier_removed": 1,
                        "interpolated_count": 0,
                        "outlier_details_(idx,val)": "Cycle removed by removal_index.csv",
                        "nan_indices": ""
                    })
                    continue

                # 위에서 걸러지지 않은 사이클만 정제 수행
                batch_cleaned[cid][cyc] = clean_physical_data(
                    cell.data[cyc],
                    cid,
                    cyc,
                    batch_reports
                )

        # Pickle 저장 (구조 동일)
        with open(batch_cache_path, "wb") as f:
            pickle.dump(
                batch_cleaned,
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )

        report_part_path = CLEAN_CACHE_DIR / f"batch_clean_{batch_id}_report.pkl"
        with open(report_part_path, "wb") as f:
            pickle.dump(
                batch_reports,
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )

        all_cleaning_reports.extend(batch_reports)

        del batch_cleaned, batch_reports
        gc.collect()

    # 리포트 CSV 저장 (구조 동일)
    report_columns = [
        'cell_id',
        'cycle',
        'feature',
        'outlier_removed',
        'interpolated_count',
        'outlier_details_(idx,val)',
        'nan_indices'
    ]

    df_report = pd.DataFrame(
        all_cleaning_reports,
        columns=report_columns
    )

    df_report.to_csv(CLEAN_REPORT_PATH, index=False)

    print(f"[Phase 1] Process finished. Report saved to: {CLEAN_REPORT_PATH}")

else:
    print(f"[Phase 1] Skipping. Using existing cache.")

## Phase 1.1 senario-based csv generating (check)

### 이상치 csv 시각화

In [ ]:
# =============================================================================
# [Verification] Phase 1 Global Removal 검증
# =============================================================================

import pandas as pd
import pickle
import os

print("--- 검증 시작 ---")

# 0. 경로 설정 (Phase 1 셀의 변수를 그대로 사용한다고 가정)
# REMOVAL_PATH, CLEAN_REPORT_PATH, CLEAN_CACHE_DIR, DATASET_TYPE,
# get_batch_id 등이 정의되어 있어야 함

_removal_path = (
    REMOVAL_PATH
    if 'REMOVAL_PATH' in locals()
    else f"../../case{HYPERPARAMS['major_version']}_{DATASET_TYPE}_removal_index.csv"
)

if not os.path.exists(_removal_path):
    print(f"{_removal_path} 파일이 없어 검증을 수행할 수 없습니다.")

elif do_remove is False:
    print("'do_remove' 옵션이 False로 설정되어 있습니다. Global Removal 검증이 스킵됩니다.")

elif START_PHASE <= 1:
    # ---------------------------------------------------------
    # [Step 1] Report CSV 수치 검증
    # ---------------------------------------------------------
    print("\n[Step 1] Report 내역 개수 검증")

    removal_df = pd.read_csv(_removal_path)

    # 원본 파일에서 중복(cell, cycle) 제거한 순수 제거 대상 개수
    expected_set = set(
        zip(
            removal_df['cell_id'],
            removal_df['cycle'].astype(int)
        )
    )
    expected_count = len(expected_set)

    try:
        report_df = pd.read_csv(CLEAN_REPORT_PATH)

        actual_removed_df = report_df[
            report_df['feature'] == "GLOBAL_REMOVAL"
        ]

        actual_set = set(
            zip(
                actual_removed_df['cell_id'],
                actual_removed_df['cycle'].astype(int)
            )
        )

        actual_count = len(actual_set)

        print(f"  - Target(목표 제거 개수): {expected_count}")
        print(f"  - Actual(실제 Report 기록 개수): {actual_count}")

        if expected_count == actual_count:
            print(
                "  [PASS] 목표 개수와 실제 리포트 기록 개수가 일치합니다."
            )
        else:
            print(
                "  [INFO] 개수가 다릅니다. "
                "(데이터셋에 없는 사이클이 removal.csv에 포함되었을 수 있습니다.)"
            )

    except Exception as e:
        print(f" Report 로드 실패: {e}")

    # ---------------------------------------------------------
    # [Step 2 & 3] Pickle 데이터 구조 정밀 검증
    # ---------------------------------------------------------
    print("\n[Step 2] 캐시(.pkl) 내 실제 삭제 여부 교차 검증")

    # 누락된 데이터 확인
    # (목표에는 있는데 실제 Report에는 없는 경우)
    missing_in_data = expected_set - actual_set

    if missing_in_data:
        print(
            f"  ℹ️ 데이터셋에 존재하지 않아 스킵된 사이클: "
            f"{len(missing_in_data)}개"
        )

    # Report에 기록된 제거 데이터 중
    # 랜덤으로 최대 3개를 뽑아서 Pickle 구조 검사
    if actual_count > 0:
        samples_to_check = list(actual_set)[:3]
        success_count = 0

        for s_cid, s_cyc in samples_to_check:
            # 해당 셀의 배치 ID 찾기
            b_id = get_batch_id(s_cid, DATASET_TYPE)

            cache_path = CLEAN_CACHE_DIR / f"batch_clean_{b_id}.pkl"

            if os.path.exists(cache_path):
                with open(cache_path, "rb") as f:
                    batch_data = pickle.load(f)

                # 검증:
                # batch_data[cid] 안에 cyc 키가 없어야 정상
                if (
                    s_cid in batch_data
                    and s_cyc in batch_data[s_cid]
                ):
                    print(
                        f" [FAIL] 셀({s_cid}), "
                        f"사이클({s_cyc})이 "
                        f"{cache_path.name} 파일에 남아있습니다!"
                    )
                else:
                    success_count += 1

            else:
                print(
                    f" 캐시 파일을 찾을 수 없습니다: {cache_path}"
                )

        if success_count == len(samples_to_check):
            print(
                f" [PASS] 랜덤 샘플({len(samples_to_check)}개) "
                f"검사 결과, pkl 파일에서 완벽히 삭제됨을 확인했습니다."
            )

    else:
        print(
            "  ⚠️ Report에 'GLOBAL_REMOVAL' 항목이 없습니다. "
            "로직이 작동하지 않았거나 대상이 없습니다."
        )

print("\n--- 검증 완료 ---")

### 전류적산 - capacity features 

In [ ]:
# import pickle
# import os
# from collections import defaultdict
# import numpy as np
# import pandas as pd
# from tqdm.auto import tqdm

def check_capacity_consistency(all_cells, dataset_type="hust"):
    """
    1) 적산 오차 5% 초과 또는 2) 용량 절대치 1.4Ah 초과 사이클을 식별하여
    세정 보고서 업데이트 및 캐시 파일(.pkl)에서 물리적으로 삭제합니다.
    """
    results = []

    for cid, cell in tqdm(all_cells.items(), desc=f"Verifying {dataset_type.upper()} Capacity"):

        cycs = sorted(cell.data.keys()) if dataset_type == "hust" \
            else sorted([int(k) for k in cell.cycles.keys() if int(k) > 0])

        for cyc in cycs:

            # 1. 메타데이터 용량 가져오기 (Ah)
            if dataset_type == "hust":
                summary_q = cell.dq.get(cyc, 0.0) / 1000.0
                df = cell.data[cyc]

            else:  # mit
                if cyc == 0:
                    continue

                summary_q = cell.summary['QD'][cyc - 1] \
                    if (cyc - 1) < len(cell.summary['QD']) else 0.0

                exclude_keys = ['Qdlin', 'Tdlin', 'dQdV']
                cycle_dict = {
                    k: v for k, v in cell.cycles[str(cyc)].items()
                    if k not in exclude_keys
                }

                df = pd.DataFrame(cycle_dict)

            if summary_q <= 0:
                continue

            # 2. 전류 적산 계산
            if dataset_type == "hust":
                t = df['Time (s)'].values
                current = df['Current (A)'].values
                dt = np.diff(t, prepend=t[0])
                dis_mask = current < -0.0001
                integrated_q = np.sum(
                    np.abs(current[dis_mask]) * dt[dis_mask]
                ) / 3600.0

            else:  # mit
                t = df['t'].values
                current = df['I'].values
                dt = np.diff(t, prepend=t[0])
                dis_mask = current < -0.0001
                integrated_q = np.sum(
                    np.abs(current[dis_mask]) * dt[dis_mask]
                ) / 55

            # 3. 오차 계산
            abs_err = abs(summary_q - integrated_q)
            rel_err = (abs_err / summary_q) * 100 if summary_q > 0 else 0

            results.append({
                "cell_id": cid,
                "cycle": cyc,
                "summary_q": summary_q,
                "integrated_q": integrated_q,
                "abs_err_ah": abs_err,
                "rel_err_pct": rel_err
            })

    report_df = pd.DataFrame(results)

    # 이상치 판정
    is_consistency_outlier = report_df['rel_err_pct'] > 5.0
    is_magnitude_outlier = report_df['summary_q'] > 1.4

    critical_df = report_df[
        is_consistency_outlier | is_magnitude_outlier
    ].copy()

    if not critical_df.empty:
        # 1. 캐시(.pkl) 파일에서 삭제
        if 'CLEAN_CACHE_DIR' in globals():
            critical_df['batch_id'] = critical_df['cell_id'].apply(
                lambda cid: get_batch_id(cid, dataset_type)
            )
            grouped = critical_df.groupby('batch_id')
            for batch_id, group in grouped:
                batch_cache_path = CLEAN_CACHE_DIR / f"batch_clean_{batch_id}.pkl"
                if not batch_cache_path.exists():
                    continue

                print(
                    f"Updating cache {batch_cache_path.name}: "
                    f"removing {len(group)} anomalous cycles..."
                )
                with open(batch_cache_path, "rb") as f:
                    batch_cleaned = pickle.load(f)
                removed_count = 0

                for _, row in group.iterrows():
                    cid, cyc = row['cell_id'], row['cycle']
                    if cid in batch_cleaned and cyc in batch_cleaned[cid]:
                        del batch_cleaned[cid][cyc]
                        removed_count += 1

                if removed_count > 0:
                    with open(batch_cache_path, "wb") as f:
                        pickle.dump(
                            batch_cleaned,
                            f,
                            protocol=pickle.HIGHEST_PROTOCOL
                        )
                    print(
                        f"Successfully removed {removed_count} cycles "
                        f"from {batch_cache_path.name}"
                    )

        # 2. 세정 보고서 업데이트
        if 'CLEAN_REPORT_PATH' in globals():
            existing_report = (
                pd.read_csv(CLEAN_REPORT_PATH)
                if os.path.exists(CLEAN_REPORT_PATH)
                else pd.DataFrame(columns=[
                    'cell_id',
                    'cycle',
                    'feature',
                    'outlier_removed',
                    'outlier_details'
                ])
            )
            
            def get_outlier_reason(row):
                reasons = []
                if row['rel_err_pct'] > 5.0:
                    reasons.append("Consistency(>5%)")
                if row['summary_q'] > 1.4:
                    reasons.append("Magnitude(>1.4Ah)")
                return " & ".join(reasons)

            new_anomalies = pd.DataFrame({
                'cell_id': critical_df['cell_id'],
                'cycle': critical_df['cycle'],
                'feature': critical_df.apply(get_outlier_reason, axis=1),
                'outlier_removed': True,
                'outlier_details': critical_df.apply(
                    lambda r:
                    f"RelErr:{r.rel_err_pct:.2f}%,"
                    f"SumQ:{r.summary_q:.3f},"
                    f"IntQ:{r.integrated_q:.3f}",
                    axis=1
                )
            })

            final_report = pd.concat(
                [existing_report, new_anomalies],
                ignore_index=True
            ).drop_duplicates(
                subset=['cell_id', 'cycle', 'feature'],
                keep='last'
            )

            final_report.to_csv(CLEAN_REPORT_PATH, index=False)
            print("Cleaning report updated with capacity anomalies.")

    # 결과 출력
    if not report_df.empty:

        print(f"\n=== {dataset_type.upper()} Capacity Consistency Report ===")
        print(f"Mean Relative Error: {report_df['rel_err_pct'].mean():.2f} %")
        print(f"Cycles > 1.4Ah: {is_magnitude_outlier.sum()}")
        print(f"Total Cycles Removed: {len(critical_df)}")

    report_df.to_csv(
        PROCESSED_DATA_ROOT / f"{dataset_type}_capacity_consistency_report.csv",
        index=False
    )

    return report_df

if START_PHASE <= 1:
    check_capacity_consistency(all_cells, dataset_type=DATASET_TYPE)

## Phase 2: Scenario-based Slicing Definition

In [ ]:
def phase2_slice_data(df, start_p=0.0, end_p=1.0, mode="C", target_q=1.1): 
    col_i = COL_I if COL_I in df.columns else [c for c in df.columns if "Current" in c][0]
    col_t = COL_TIME
    
    # 1. Select data for the mode (Charge or Discharge)
    mode_df = df[df[col_i] > 0.01].copy() if mode == "C" else df[df[col_i] < -0.01].copy()
    if mode_df.empty: return mode_df, 0, (1 if mode == "C" else 0)
    
    # 2. Calculate relative accumulated capacity (Ah) for this specific phase
    if 'capacity' in mode_df.columns:
        q_acc = (mode_df['capacity'] - mode_df['capacity'].iloc[0]).abs()
    else:
        dt = mode_df[col_t].diff().fillna(0)
        dq = (mode_df[col_i].abs() * dt) / 3600.0
        q_acc = dq.cumsum()
        
    # 3. Calculate capacity ratio relative to current cycle's capacity (target_q)
    q_ratio = q_acc / (target_q + 1e-9)
    q_ratio = np.clip(q_ratio, 0.0, 1.05)
    
    # 4. Slicing based on capacity bounds
    mask = (q_ratio >= start_p) & (q_ratio <= end_p)
    sliced = mode_df[mask]
    
    # 5. Labeling based on Mode (Charge vs Discharge)
    avg_pos = (start_p + end_p) / 2
    mode_label = 1 if mode == "C" else 0
    
    if mode == "D":
        # 방전 (Discharge): 누적 0.0~0.3 = 방전 초기 = SOC 100~70% (High)
        if avg_pos <= 0.3:
            soc_label = -2  # High
        elif avg_pos <= 0.7:
            soc_label = -1  # Mid
        else:
            soc_label = 0   # Low
    else:
        # 충전 (Charge): 누적 0.0~0.3 = 충전 초기 = SOC 0~30% (Low)
        if avg_pos <= 0.3:
            soc_label = 0   # Low 
        elif avg_pos <= 0.7:
            soc_label = -1  # Mid
        else:
            soc_label = -2  # High 

    return sliced, soc_label, mode_label

## Phase 2.1: Global Slicing Loop

In [ ]:
scen_bounds = {"H": (0.0, 0.3), "M": (0.3, 0.7), "L": (0.7, 1.0)}
lengths = [0.2, 0.3, 0.4]
step = 0.1

In [ ]:
all_raw_segments = []
batch_groups = defaultdict(dict)

if DATASET_TYPE == "mit" and START_PHASE >= 3 and (CACHE_DIR / f"batch_b3.pkl").exists():
    print(f"[Phase 2] Skipping. Loading existing sliced segments from cache...")
    # Load all batch files for MIT
    for batch_file in tqdm(sorted(CACHE_DIR.glob("batch_b*.pkl"))):
        with open(batch_file, "rb") as f: batch_data = pickle.load(f)
        all_raw_segments.extend(batch_data['segments'])
    
elif DATASET_TYPE == "hust" and START_PHASE >= 3 and (CACHE_DIR / f"batch_10.pkl").exists():
    print(f"[Phase 2] Skipping. Loading existing sliced segments from cache...")
    # Load all batch files for HUST
    for batch_file in tqdm(sorted(CACHE_DIR.glob("batch_*.pkl"))):
        with open(batch_file, "rb") as f: batch_data = pickle.load(f)
        all_raw_segments.extend(batch_data['segments'])

else:
    for cid, cell in all_cells.items():
        b_id = get_batch_id(cid, DATASET_TYPE)
        batch_groups[b_id][cid] = cell

    for batch_id in tqdm(sorted(batch_groups.keys())):
        slice_cache_path = CACHE_DIR / f"batch_{batch_id}.pkl"
        clean_cache_path = CLEAN_CACHE_DIR / f"batch_clean_{batch_id}.pkl"
        if slice_cache_path.exists():
            with open(slice_cache_path, "rb") as f: batch_data = pickle.load(f)
            all_raw_segments.extend(batch_data['segments'])
            continue

        with open(clean_cache_path, "rb") as f: batch_cleaned = pickle.load(f)
        # --- Sampling Strategy for Volume Reduction ---
        batch_segments = []
        for cid, cell in tqdm(batch_groups[batch_id].items(), desc=f"Processing Batch {batch_id}"):
            cleaned_cycles = batch_cleaned.get(cid, {})
            
            # Get nominal capacity for this cell (e.g., first valid cycle capacity)
            nominal_q = 1.1 # Default
            if cleaned_cycles:
                sorted_all = sorted(cleaned_cycles.keys())
                for test_cyc in sorted_all:
                    _, test_cap = get_cell_labels(cell, test_cyc, DATASET_TYPE)
                    if test_cap > 0:
                        nominal_q = test_cap
                        break
                
            # 1. Apply CYCLE_STEP to sample cycles (Skip Cycle 1 as it often has 0 capacity)
            # sorted_cycs = sorted(cleaned_cycles.keys())
            # sampled_cycs = sorted_cycs[1:][::CYCLE_STEP]  
            
            # 정렬 방식을 숫자로 강제 (MIT 문자열 키 대응)
            sorted_cycs = sorted(cleaned_cycles.keys(), key=lambda x: int(x))
            sampled_cycs = sorted_cycs[1:][::cycle_step]  # Skip Cycle 1 and apply step                                                         
            
            for cyc in tqdm(sampled_cycs, desc=f"  Slicing Cell {cid}"):
                df = cleaned_cycles[cyc]
                num_cols = df.select_dtypes(include=[np.number]).columns
                df[num_cols] = df[num_cols].astype(np.float32)
                
                rul_label, cap_label = get_cell_labels(cell, cyc, DATASET_TYPE)
                
                for mode in ["C", "D"]:
                    for scen_name, (s_bound, e_bound) in scen_bounds.items():
                        for length in lengths:
                            max_start = e_bound - length
                            if max_start < s_bound: continue
                            
                            # 2. Use SLIDING_STEP for spatial sampling
                            for start_p in np.arange(s_bound, max_start + 0.0001, sliding_step):
                                end_p = start_p + length
                                # Pass nominal_q as target_q for SOC-based slicing
                                sliced, soc_l, mode_l = phase2_slice_data(df, start_p, end_p, mode, target_q=cap_label)
                                if not sliced.empty and len(sliced) >= 5:
                                    batch_segments.append({
                                        "cell": cid, "cyc": cyc, "df": sliced,
                                        "soc_label": soc_l, "mode_label": mode_l,
                                        "length_p": length,
                                        "rul": rul_label,
                                        "capacity": cap_label,
                                        "cycle_key": (cid, cyc)
                                    })

        with open(slice_cache_path, "wb") as f:
            pickle.dump({'segments': batch_segments}, f, protocol=pickle.HIGHEST_PROTOCOL)

        all_raw_segments.extend(batch_segments)
        del batch_cleaned, batch_segments
        gc.collect()
        print(f"Total segments: {len(all_raw_segments)}")

## Phase 2.2: Slicing Summary Visualization

In [ ]:
# ============================================================
# Segment Visualization: All segments for a single cell & cycle
# Shows longest segment per (scenario, mode) group as continuous lines
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# --- Configuration ---
TARGET_CELL = list(all_cells.keys())[0]  # Change to desired cell ID
TARGET_CYC = None  # None = auto-pick first available cycle with segments

# --- Filter segments for the target cell ---
cell_segments = [s for s in all_raw_segments if s['cell'] == TARGET_CELL]
if TARGET_CYC is None:
    available_cycs = sorted(set(s['cyc'] for s in cell_segments), key=lambda x: int(x))
    TARGET_CYC = available_cycs[0]
    print(f"Auto-selected cycle: {TARGET_CYC}")

cyc_segments = [s for s in cell_segments if s['cyc'] == TARGET_CYC]
print(f"Cell={TARGET_CELL}, Cycle={TARGET_CYC}: {len(cyc_segments)} total segments")

# --- Keep only the longest segment per (mode_label, soc_label) group ---
# Group by (mode_label, soc_label) and keep the one with max rows
from collections import defaultdict
group_best = {}
for seg in cyc_segments:
    key = (seg['mode_label'], seg['soc_label'])
    if key not in group_best or len(seg['df']) > len(group_best[key]['df']):
        group_best[key] = seg

selected_segments = list(group_best.values())
print(f"After keeping longest per (mode, soc_label): {len(selected_segments)} segments")

# --- Get the full cycle data for background reference ---
# Try loading from cleaned cache
b_id = get_batch_id(TARGET_CELL, DATASET_TYPE)
clean_cache_path = CLEAN_CACHE_DIR / f"batch_clean_{b_id}.pkl"
full_cycle_df = None
if clean_cache_path.exists():
    import pickle
    with open(clean_cache_path, "rb") as f:
        batch_cleaned = pickle.load(f)
    if TARGET_CELL in batch_cleaned and TARGET_CYC in batch_cleaned[TARGET_CELL]:
        full_cycle_df = batch_cleaned[TARGET_CELL][TARGET_CYC]
    del batch_cleaned

# --- Color mapping ---
# mode_label: 1=Charge(C), 0=Discharge(D)
# soc_label: -2=High, -1=Mid, 0=Low
soc_names = {-2: "High", -1: "Mid", 0: "Low"}
mode_names = {1: "Charge", 0: "Discharge"}

# Distinct colors per (mode, soc) group
color_map = {
    (1, -2): "#e74c3c",   # Charge-High  (red)
    (1, -1): "#e67e22",   # Charge-Mid   (orange)
    (1,  0): "#f1c40f",   # Charge-Low   (yellow)
    (0, -2): "#2980b9",   # Discharge-High (blue)
    (0, -1): "#27ae60",   # Discharge-Mid  (green)
    (0,  0): "#8e44ad",   # Discharge-Low  (purple)
}

# --- Plot ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
fig.suptitle(f"Segments for Cell={TARGET_CELL}, Cycle={TARGET_CYC}\n"
                f"(Longest per mode×SOC group, {len(selected_segments)} segments)",
                fontsize=14, fontweight='bold')

# Plot full cycle as light background if available
if full_cycle_df is not None:
    row_idx_full = np.arange(len(full_cycle_df))
    time_full = full_cycle_df[COL_TIME].values if COL_TIME in full_cycle_df.columns else row_idx_full
    ax1.plot(row_idx_full, full_cycle_df[COL_V].values, color='#cccccc', lw=1.5, 
                zorder=0, label='Full cycle')
    ax2.plot(row_idx_full, full_cycle_df[COL_I].values, color='#cccccc', lw=1.5, 
                zorder=0, label='Full cycle')

# Plot each selected segment
legend_handles = []
for seg in sorted(selected_segments, key=lambda s: s['df'].index[0]):
    ml, sl = seg['mode_label'], seg['soc_label']
    color = color_map.get((ml, sl), '#333333')
    label = f"{mode_names[ml]}-{soc_names[sl]} (len={seg['length_p']:.0%})"
    
    df_seg = seg['df']
    
    # Use original index positions as row numbers within full cycle
    if full_cycle_df is not None:
        # Segment df preserves original index from the full cycle df
        row_positions = np.array([full_cycle_df.index.get_loc(idx) 
                                    if idx in full_cycle_df.index else i 
                                    for i, idx in enumerate(df_seg.index)])
    else:
        row_positions = np.arange(len(df_seg))
    
    ax1.plot(row_positions, df_seg[COL_V].values, color=color, lw=2.0, alpha=0.85, zorder=2)
    ax2.plot(row_positions, df_seg[COL_I].values, color=color, lw=2.0, alpha=0.85, zorder=2)
    
    legend_handles.append(mpatches.Patch(color=color, label=label))

# --- Dual x-axis (row index + time) ---
ax2.set_xlabel("Row Index (within cycle)", fontsize=12)

if full_cycle_df is not None and COL_TIME in full_cycle_df.columns:
    time_vals = full_cycle_df[COL_TIME].values
    ax_time = ax2.twiny()
    ax_time.set_xlim(ax2.get_xlim())
    # Set time ticks at evenly spaced row positions
    n_ticks = 8
    tick_rows = np.linspace(0, len(full_cycle_df) - 1, n_ticks, dtype=int)
    tick_times = time_vals[tick_rows]
    ax_time.set_xticks(tick_rows)
    ax_time.set_xticklabels([f"{t:.0f}s" for t in tick_times], fontsize=9)
    ax_time.set_xlabel("Time (s)", fontsize=12)
    # Move twin axis to bottom
    ax_time.xaxis.set_ticks_position('bottom')
    ax_time.xaxis.set_label_position('bottom')
    ax_time.spines['bottom'].set_position(('outward', 36))

ax1.set_ylabel("Voltage (V)", fontsize=12)
ax2.set_ylabel("Current (A)", fontsize=12)
ax1.grid(True, alpha=0.3)
ax2.grid(True, alpha=0.3)
ax1.legend(handles=legend_handles, loc='upper right', fontsize=9, ncol=2)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.savefig(figure_path / f"segments_viz_{TARGET_CELL}_cyc{TARGET_CYC}.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to: {figure_path / f'segments_viz_{TARGET_CELL}_cyc{TARGET_CYC}.png'}")

In [ ]:
def plot_phase2_slicing_summary(segments, num_samples=3):
    # 1. 만약 segments가 비어있다면 디스크 캐시에서 '모든' 배치 파일 로드
    if not segments:
        print(f"[Info] 'segments'가 비어있습니다. {CACHE_DIR}에서 모든 배치를 로드합니다...")
        cache_files = sorted(CACHE_DIR.glob("batch_*.pkl"))
        if not cache_files:
            print(f"[Error] 캐시된 슬라이싱 배치 파일이 없습니다 ({CACHE_DIR}). Phase 2를 먼저 실행하세요.")
            return
        
        all_loaded_segments = []
        for cf in tqdm(cache_files, desc="Loading All Sliced Batches"):
            with open(cf, "rb") as f:
                batch_data = pickle.load(f)
                # 각 배치 파일 내부의 'segments' 리스트를 통합
                all_loaded_segments.extend(batch_data.get('segments', []))
        
        segments = all_loaded_segments
        print(f"총 {len(cache_files)}개의 배치 파일에서 {len(segments):,}개의 세그먼트를 모두 로드했습니다.")

    if not segments:
        print("[Error] 시각화할 데이터가 없습니다.")
        return

    # 2. 데이터프레임 변환 (전체 데이터 대상)
    # 샘플링 없이 range(len(segments))를 사용하여 전체 메타데이터 생성
    print(f"[Info] 전체 {len(segments):,}개 세그먼트에 대한 통계 분석 중...")
    
    df_meta = pd.DataFrame([{
        "soc": s["soc_label"],
        "mode": s["mode_label"],
        "len": len(s["df"]),
        "cap": s["capacity"]
    } for s in segments])

    # 3. 시각화 로직 (전체 분포 반영)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # SOC 라벨 및 충/방전 분포 (전체)
    sns.countplot(data=df_meta, x='soc', hue='mode', ax=axes[0,0])
    axes[0,0].set_title(
        f"Full Dataset: SOC Labels Distribution\n"
        f"(-2:H, -1:M, 0:L | Total: {len(df_meta):,})"
    )
    
    # 세그먼트 길이 분포 (전체)
    sns.histplot(data=df_meta, x='len', ax=axes[0,1], kde=True)
    axes[0,1].set_title("Full Dataset: Segment Sample Lengths")
    axes[0,1].set_xlim(0, 600)
    
    # 용량 분포 (전체)
    df_meta_nonzero = df_meta[df_meta['cap'] > 0]
    if not df_meta_nonzero.empty:
        sns.boxplot(data=df_meta_nonzero, x='soc', y='cap', ax=axes[1,0])
        axes[1,0].set_title("Full Dataset: Capacity Distribution per Scenario")
    
    # 샘플 시각화 (랜덤 하나)
    sample_s = random.choice(segments)
    cid, cyc = sample_s['cell'], sample_s['cyc']
    
    try:
        if 'all_cells' in globals() and cid in all_cells:
            full_df = all_cells[cid].data[cyc]
            
            axes[1,1].plot(
                full_df[COL_V].values,
                color='gray',
                alpha=0.3,
                label='Full Cycle'
            )
            
            axes[1,1].plot(
                sample_s['df'].index % len(full_df),
                sample_s['df'][COL_V].values,
                color='red',
                linewidth=2,
                label='Sliced Segment'
            )
            
            axes[1,1].set_title(f"Example: Cell {cid} Cyc {cyc}")
            axes[1,1].legend()
            
    except Exception as e:
        axes[1,1].text(
            0.5,
            0.5,
            f"Original data plot skipped\n({e})",
            ha='center'
        )
    
    plt.tight_layout()
    plt.savefig(FIGURE_ROOT / f"{DATASET_TYPE}_phase2_full_slicing_summary.png")
    plt.show()

if START_PHASE >= 3:
    print(f"{CACHE_DIR} 경로에서 저장된 세그먼트 파일을 불러옵니다...")
    all_raw_segments = []
    cache_files = sorted(CACHE_DIR.glob("batch_*.pkl"))
    for cf in tqdm(cache_files, desc="Loading Sliced Batches for Summary"):
        with open(cf, "rb") as f:
            batch_data = pickle.load(f)
            all_raw_segments.extend(batch_data.get('segments', []))
    print(f"총 {len(cache_files)}개의 배치 파일에서 {len(all_raw_segments):,}개의 세그먼트를 로드했습니다.")

plot_phase2_slicing_summary(all_raw_segments)

In [ ]:


import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import savgol_filter

def plot_random_segments_sample_with_derivatives(segments, num_samples=3):
    """
    세그먼트 데이터풀에서 무작위로 샘플을 뽑아 6개의 물리량
    (V, I, T, Time, IC(dQ/dV), DVA(dV/dQ))을 시각화합니다.
    """
    if not segments:
        print("시각화할 세그먼트 데이터가 없습니다.")
        return

    selected_samples = []
    candidates = random.sample(segments, min(len(segments), num_samples * 10))

    seen_cells = set()
    for cand in candidates:
        if cand['cell'] not in seen_cells:
            selected_samples.append(cand)
            seen_cells.add(cand['cell'])
        if len(selected_samples) == num_samples:
            break

    if len(selected_samples) < num_samples:
        selected_samples = random.sample(segments, num_samples)

    fig, axes = plt.subplots(3, 2, figsize=(16, 18))
    plt.subplots_adjust(hspace=0.35, wspace=0.2)
    axes = axes.flatten()

    titles = [
        'Voltage (V)', 'Current (A)',
        'Temperature (°C)', 'Time (s)',
        'Incremental Capacity (dQ/dV)', 'Differential Voltage (dV/dQ)'
    ]

    col_map = {
        'voltage': ['voltage', 'Voltage (V)', 'V'],
        'current': ['current', 'Current (A)', 'I'],
        'temp': ['temperature', 'Temperature (degC)', 'T', COL_T],
        'time': ['time', 'Time (s)', 't'],
        'capacity': ['capacity', 'Capacity (Ah)', 'Q', 'Qc', 'Qd']
    }

    def get_col(df, key):
        for c in col_map[key]:
            if c in df.columns:
                return c
        return None

    # colors = ['dodgerblue', 'crimson', 'forestgreen']
    colors = sns.color_palette("Set2", num_samples)

    for i, sample in enumerate(selected_samples):
        # pdb.set_trace()
        # sample이 charge면 건너뛰기
        if sample['mode_label'] == 0:
            continue
        
        df = sample['df'].reset_index(drop=True)

        soc_map = {-2: "High", -1: "Mid", 0: "Low"}
        mode_map = {1: "Charge", 0: "Discharge"}

        label = (
            f"Cell: {sample['cell']}, Cyc: {sample['cyc']}, "
            f"{mode_map.get(sample['mode_label'], '??')}-"
            f"{soc_map.get(sample['soc_label'], '??')}"
        )

        target_keys = ['voltage', 'current', 'temp', 'time']
        for j, key in enumerate(target_keys):
            col_name = get_col(df, key)
            if col_name:
                axes[j].plot(df.index, df[col_name], label=label, color=colors[i], linewidth=2)
                axes[j].set_title(titles[j], fontsize=14, fontweight='bold')
                axes[j].grid(True, linestyle=':', alpha=0.6)
                axes[j].legend(fontsize=9, loc='best')
            else:
                axes[j].text(0.5, 0.5, f"Column for {key} not found", ha='center')

        v_col = get_col(df, 'voltage')
        q_col = get_col(df, 'capacity')

        if v_col and q_col and len(df) > 11:
            try:
                v_smooth = savgol_filter(df[v_col], window_length=11, polyorder=3)
                q_smooth = savgol_filter(df[q_col], window_length=11, polyorder=3)

                dV = np.gradient(v_smooth)
                dQ = np.gradient(q_smooth)

                dV_safe = np.where(dV == 0, 1e-6, dV)
                dQ_safe = np.where(dQ == 0, 1e-6, dQ)

                dQ_dV = dQ / dV_safe
                dV_dQ = dV / dQ_safe

                dQ_dV_clip = np.clip(dQ_dV, -50, 50)
                dV_dQ_clip = np.clip(dV_dQ, -10, 10)

                axes[4].plot(v_smooth, dQ_dV_clip, label=label, color=colors[i], linewidth=2)
                axes[4].set_title(titles[4], fontsize=14, fontweight='bold')
                axes[4].set_xlabel("Voltage (V)")
                axes[4].set_ylabel("dQ/dV")
                axes[4].grid(True, linestyle=':', alpha=0.6)
                axes[4].legend(fontsize=9, loc='best')

                axes[5].plot(q_smooth, dV_dQ_clip, label=label, color=colors[i], linewidth=2)
                axes[5].set_title(titles[5], fontsize=14, fontweight='bold')
                axes[5].set_xlabel("Capacity (Ah)")
                axes[5].set_ylabel("dV/dQ")
                axes[5].grid(True, linestyle=':', alpha=0.6)
                axes[5].legend(fontsize=9, loc='best')

            except Exception as e:
                axes[4].text(0.5, 0.5, "IC Calc Error", ha='center')
                axes[5].text(0.5, 0.5, "DVA Calc Error", ha='center')
        else:
            axes[4].text(0.5, 0.5, "Not enough V/Q data", ha='center')
            axes[5].text(0.5, 0.5, "Not enough V/Q data", ha='center')

    plt.suptitle(
        f"Random Segments Inspection & Electrochemical Analysis (n={num_samples})",
        fontsize=20,
        y=0.98
    )

    if 'FIGURE_ROOT' in globals():
        save_path = Path(FIGURE_ROOT) / f"{DATASET_TYPE}_segment_inspection_derivatives.png"
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f">> 샘플 시각화 결과가 저장되었습니다: {save_path}")

    plt.show()

# 함수 호출 예시
plot_random_segments_sample_with_derivatives(all_raw_segments, 7)

## Phase 3: 45D Adaptive HI Extraction Logic (45D Base + 30D New)

In [ ]:
from scipy.stats import skew, kurtosis, pearsonr

def extract_75d_hi(df, mode="C"):
    v_col = COL_V
    i_col = COL_I
    t_col = COL_TIME
    temp_col = COL_T
    
    v = df[v_col].values if v_col in df.columns else df.iloc[:, 1].values
    i_raw = df[i_col].values if i_col in df.columns else df.iloc[:, 2].values
    t = df[t_col].values if t_col in df.columns else df.iloc[:, 0].values
    temp = df[temp_col].values if temp_col in df.columns else np.full_like(v, 25.0)
    
    i_abs = np.abs(i_raw)
    dt_full = np.diff(t, prepend=t[0])
    dt_full = np.where(dt_full > 0, dt_full, 1e-6)
    dt_diff = np.where(np.diff(t) > 0, np.diff(t), 1e-6)
    dq_instant = (i_abs[1:] * dt_diff) / 3600.0
    dq_cum = np.cumsum(np.insert(dq_instant, 0, 0))
    his = {}
    
    # ========================================================
    # 1. Base 45D Features (Original Logic)
    # ========================================================
    his["c1_1_mean_v"] = np.mean(v)
    his["c1_2_var_v"] = np.var(v)
    his["c1_3_skew_v"] = skew(v) if np.std(v) > 1e-6 else 0.0
    his["c1_4_kurt_v"] = kurtosis(v) if np.std(v) > 1e-6 else 0.0
    his["c1_5_mean_t"] = np.mean(temp)
    his["c1_6_delta_t"] = np.max(temp) - np.min(temp)
    his["c1_7_dtdt"] = np.mean(np.diff(temp) / dt_diff)
    his["c1_8_mean_i"] = np.mean(i_abs)
    his["c1_9_var_i"] = np.var(i_abs)

    p_v = np.histogram(v, bins=10, density=True)[0]
    his["c1_10_ent_v"] = -np.sum(p_v * np.log(p_v + 1e-6))
    p_t = np.histogram(temp, bins=10, density=True)[0]
    his["c1_11_ent_t"] = -np.sum(p_t * np.log(p_t + 1e-6))

    his["c1_12_corr_vi"] = pearsonr(v, i_abs)[0] if np.std(v) > 1e-6 and np.std(i_abs) > 1e-6 else 0.0
    his["c1_13_corr_vt"] = pearsonr(v, temp)[0] if np.std(v) > 1e-6 and np.std(temp) > 1e-6 else 0.0
    his["c1_14_de_dq"] = np.sum(v * i_abs * dt_full) / (np.sum(i_abs * dt_full) + 1e-6)
    his["c1_15_power_var"] = np.var(v * i_raw)
    
    for j in range(1, 16): his[f"c2_dis_{j}"] = his[f"c3_cha_{j}"] = 0.0
    dvdt = np.diff(v) / dt_diff
    d2vdt2 = np.diff(dvdt) / np.where(dt_diff[1:] > 0, dt_diff[1:], 1e-6) if len(dvdt) > 1 else np.array([0.0])

    if mode == "D":
        his["c2_dis_1"] = np.mean(dvdt); 
        his["c2_dis_2"] = np.var(dvdt); his["c2_dis_3"] = np.mean(d2vdt2)
        his["c2_dis_4"] = np.sum((np.max(v) - v) * dt_full)
        di = np.diff(i_raw); valid_di = np.abs(di) > 0.005
        his["c2_dis_5"] = np.mean(np.abs(np.diff(v)[valid_di] / di[valid_di])) if np.any(valid_di) else 0.0
        his["c2_dis_6"] = np.sum(v * i_abs * dt_full) / 3600.0
        his["c2_dis_7"] = np.sum(i_abs * dt_full) / 3600.0
        dvdq_base = np.diff(v) / np.where(dq_instant > 0, dq_instant, 1e-6)
        his["c2_dis_8"] = np.mean(dvdq_base); his["c2_dis_9"] = np.var(dvdq_base)
        his["c2_dis_10"] = np.min(v) / (np.max(v) + 1e-6)
        his["c2_dis_11"] = (temp[-1] - temp[0]) / (his["c2_dis_7"] + 1e-6)
        his["c2_dis_12"] = np.sum(np.abs(np.diff(dvdt)))
        his["c2_dis_13"] = np.sum(v * dt_full) / (np.mean(i_abs) + 1e-6)
        if len(dq_cum) > 1: his["c2_dis_14"] = np.polyfit(dq_cum, v, 1)[0]
        his["c2_dis_15"] = np.sum(v * dq_cum) / (np.sum(dq_cum) + 1e-6)
        
    elif mode == "C":
        his["c3_cha_1"] = np.mean(dvdt) 
        his["c3_cha_2"] = np.var(dvdt) 
        his["c3_cha_3"] = np.mean(d2vdt2)
        his["c3_cha_4"] = np.sum((v - np.min(v)) * dt_full)
        dv_pos = np.where(np.diff(v) > 0.0005, np.diff(v), 1e-6); dqdv_base = dq_instant / dv_pos
        his["c3_cha_5"] = np.mean(dqdv_base); 
        his["c3_cha_6"] = np.var(dqdv_base)
        his["c3_cha_7"] = np.sum(v * i_abs * dt_full) / 3600.0
        his["c3_cha_8"] = np.sum(i_abs * dt_full) / 3600.0
        his["c3_cha_9"] = np.max(v) / (np.min(v) + 1e-6)
        his["c3_cha_10"] = (temp[-1] - temp[0]) / (his["c3_cha_8"] + 1e-6)
        his["c3_cha_11"] = np.sum(np.abs(np.diff(dvdt)))
        his["c3_cha_12"] = np.mean(i_abs) / (np.abs(np.mean(dvdt)) + 1e-6)
        p_dvdt = np.histogram(dvdt, bins=10, density=True)[0]
        his["c3_cha_13"] = -np.sum(p_dvdt * np.log(p_dvdt + 1e-6))
        his["c3_cha_14"] = np.max(dqdv_base) if len(dqdv_base) > 0 else 0.0
        his["c3_cha_15"] = np.sum((v[1:] - v[0]) * dq_instant)

    # ========================================================
    # 2. New 30D Features (IC/DV & Plateau based, No Smoothing)
    # ========================================================
    dv = np.diff(v, prepend=v[0])
    dv_valid = np.where(np.abs(dv) > 1e-5, dv, 1e-5)
    dq = (i_abs * dt_full) / 3600.0
    dq_valid = np.where(dq > 1e-6, dq, 1e-6)
    
    ic = dq / dv_valid
    dv_dq = dv / dq_valid
    
    his["n1_ic_peak_pos"] = v[np.argmax(np.abs(ic))]
    his["n2_ic_grad_max"] = np.max(np.abs(np.diff(ic, prepend=ic[0]) / dt_full))
    his["n3_ic_width_half"] = np.std(v[np.abs(ic) > np.max(np.abs(ic))*0.5]) if np.any(ic) else 0.0
    his["n4_ic_skew"] = skew(ic) if len(ic) > 1 else 0.0
    his["n5_ic_kurt"] = kurtosis(ic) if len(ic) > 1 else 0.0
    his["n6_ic_valley"] = np.percentile(np.abs(ic), 5)
    his["n7_ic_t_corr"] = pearsonr(ic, np.diff(temp, prepend=temp[0]))[0] if np.std(temp)>1e-6 else 0.0
    his["n8_ic_peak_count"] = np.sum((np.diff(np.sign(np.diff(ic))) != 0) & (np.abs(ic[1:-1]) > np.mean(np.abs(ic))))

    dv_dq_clean = dv_dq[np.isfinite(dv_dq)]
    his["n9_dv_center_v"] = v[np.argmin(np.abs(dv_dq))] if len(dv_dq)>0 else 0.0
    his["n10_dv_min"] = np.percentile(np.abs(dv_dq_clean), 5) if len(dv_dq_clean)>0 else 0.0
    his["n11_dv_peak_gap"] = np.ptp(dq_cum[np.abs(dv_dq) > np.percentile(np.abs(dv_dq), 90)]) if len(dq_cum)>1 else 0.0
    d2v_dq2 = np.diff(dv_dq, prepend=dv_dq[0]) / dq_valid
    his["n12_dv_curvature"] = np.mean(np.abs(d2v_dq2))
    his["n13_dv_area_thresh"] = np.sum(dq[np.abs(dv_dq) < 0.1])
    his["n14_dv_symmetry"] = np.std(dv_dq[:len(dv_dq)//2]) / (np.std(dv_dq[len(dv_dq)//2:]) + 1e-6)

    plateau_mask = np.abs(dv_dq) < 0.2
    his["n15_plateau_len"] = np.sum(dq[plateau_mask])
    his["n16_v_sum_abs_diff"] = np.sum(np.abs(dv))
    his["n17_v_grad_sum"] = np.sum(np.abs(dv / dt_full))
    his["n18_plateau_start_v"] = v[np.where(plateau_mask)[0][0]] if np.any(plateau_mask) else v[0]
    his["n19_plateau_end_v"] = v[np.where(plateau_mask)[0][-1]] if np.any(plateau_mask) else v[-1]
    his["n20_imp_proxy"] = np.mean((v - 3.2) / (i_raw + 1e-6))
    his["n21_v_roughness"] = np.std(dv / dt_full)
    his["n22_v_auc_time"] = np.trapz(v - np.min(v), t)
    his["n23_nonlinear_idx"] = np.polyfit(dq_cum, v, 3)[0] if len(dq_cum) > 3 else 0.0
    his["n24_trans_slope"] = (v[1]-v[0]) / (v[-1]-v[-2] + 1e-6) if len(v) > 1 else 0.0

    his["n25_thermal_stability"] = np.std(np.diff(temp, prepend=temp[0]) / dt_full)
    dT = np.diff(temp, prepend=temp[0])
    his["n26_dq_dt_plateau"] = np.mean(dq[plateau_mask] / (np.abs(dT[plateau_mask]) + 1e-6))
    his["n27_energy_eff_ratio"] = np.sum(v * dq) / (dq_cum[-1] * 3.2 + 1e-6)
    his["n28_v_relax_rate"] = np.abs(v[-1] - v[-5]) / (t[-1] - t[-5] + 1e-6) if len(v) > 5 else 0.0
    his["n29_time_to_step"] = t[np.argmin(np.abs(v - (v[0] + 0.1)))] - t[0]
    his["n30_boundary_shift"] = v[-1] - v[0]
    
    return pd.Series(his)

In [ ]:
hi_feature_names = [
    # ==========================================================
    # Common Features (c1)
    # ==========================================================
    "c1_1_mean_v",             # Mean Voltage
    "c1_2_var_v",              # Voltage Variance
    "c1_3_skew_v",             # Voltage Skewness
    "c1_4_kurt_v",             # Voltage Kurtosis
    "c1_5_mean_t",             # Mean Temperature
    "c1_6_delta_t",            # Temperature Range
    "c1_7_dtdt",               # Mean Temperature Change Rate
    "c1_8_mean_i",             # Mean Current Magnitude
    "c1_9_var_i",              # Current Variance
    "c1_10_ent_v",             # Voltage Entropy
    "c1_11_ent_t",             # Temperature Entropy
    "c1_12_corr_vi",           # Voltage-Current Correlation
    "c1_13_corr_vt",           # Voltage-Temperature Correlation
    "c1_14_de_dq",             # Energy per Charge
    "c1_15_power_var",         # Power Variance

    # ==========================================================
    # Discharge Features (c2)
    # ==========================================================
    "c2_dis_1_mean_dvdt",              # Mean Voltage Decay Rate
    "c2_dis_2_var_dvdt",               # Variance of Voltage Decay Rate
    "c2_dis_3_mean_d2vdt2",            # Mean Voltage Curvature
    "c2_dis_4_voltage_drop_integral",  # Accumulated Voltage Drop
    "c2_dis_5_dynamic_resistance",     # Dynamic Internal Resistance Estimate
    "c2_dis_6_discharge_energy",       # Delivered Energy
    "c2_dis_7_discharge_capacity",     # Delivered Capacity
    "c2_dis_8_mean_dvdq",              # Mean DVA (dV/dQ)
    "c2_dis_9_var_dvdq",               # Variance of DVA
    "c2_dis_10_voltage_retention",     # Voltage Retention Ratio
    "c2_dis_11_temp_rise_per_ah",      # Temperature Rise per Capacity
    "c2_dis_12_voltage_fluctuation",   # Voltage Dynamics Roughness
    "c2_dis_13_voltage_efficiency",    # Voltage-Time Efficiency
    "c2_dis_14_ocv_slope",             # Voltage-Capacity Slope
    "c2_dis_15_voltage_charge_moment", # Voltage-Charge Integral

    # ==========================================================
    # Charge Features (c3)
    # ==========================================================
    "c3_cha_1_mean_dvdt",              # Mean Voltage Rise Rate
    "c3_cha_2_var_dvdt",               # Variance of Voltage Rise Rate
    "c3_cha_3_mean_d2vdt2",            # Mean Voltage Curvature
    "c3_cha_4_voltage_gain_integral",  # Accumulated Voltage Gain
    "c3_cha_5_mean_dqdv",              # Mean IC (dQ/dV)
    "c3_cha_6_var_dqdv",               # Variance of IC
    "c3_cha_7_charge_energy",          # Charged Energy
    "c3_cha_8_charge_capacity",        # Charged Capacity
    "c3_cha_9_voltage_growth_ratio",   # Max/Min Voltage Ratio
    "c3_cha_10_temp_rise_per_ah",      # Temperature Rise per Capacity
    "c3_cha_11_voltage_fluctuation",   # Voltage Dynamics Roughness
    "c3_cha_12_current_voltage_gain",  # Current-to-Voltage Gain Ratio
    "c3_cha_13_dvdt_entropy",          # Voltage Rate Entropy
    "c3_cha_14_ic_peak",               # Peak IC Value
    "c3_cha_15_charge_voltage_integral",# Voltage-Charge Integral

    # ==========================================================
    # New Physical Features (n)
    # ==========================================================
    "n1_ic_peak_pos",
    "n2_ic_grad_max",
    "n3_ic_width_half",
    "n4_ic_skew",
    "n5_ic_kurt",
    "n6_ic_valley",
    "n7_ic_t_corr",
    "n8_ic_peak_count",
    "n9_dv_center_v",
    "n10_dv_min",
    "n11_dv_peak_gap",
    "n12_dv_curvature",
    "n13_dv_area_thresh",
    "n14_dv_symmetry",
    "n15_plateau_len",
    "n16_v_sum_abs_diff",
    "n17_v_grad_sum",
    "n18_plateau_start_v",
    "n19_plateau_end_v",
    "n20_imp_proxy",
    "n21_v_roughness",
    "n22_v_auc_time",
    "n23_nonlinear_idx",
    "n24_trans_slope",
    "n25_thermal_stability",
    "n26_dq_dt_plateau",
    "n27_energy_eff_ratio",
    "n28_v_relax_rate",
    "n29_time_to_step",
    "n30_boundary_shift"
]

## Phase 3.1: Feature Extraction & Scaling

In [ ]:
final_batch_files = sorted(FEATURE_BATCH_DIR.glob("final_*.pkl"))

if START_PHASE <= 3 or not final_batch_files:
    SKIP_EXTRACTION = False
    print(f"[Phase 3] Cache not found or START_PHASE <= 3. Starting Feature Extraction for {DATASET_TYPE}...")
else:
    SKIP_EXTRACTION = True
    print(f"[Phase 3] Cache found ({len(final_batch_files)} batches). Loading from {FEATURE_BATCH_DIR}...")
    batch_feature_pool = []
    for bf in tqdm(final_batch_files, desc="Loading cached features"): 
        with open(bf, "rb") as f: batch_feature_pool.extend(pickle.load(f))


In [ ]:
if not SKIP_EXTRACTION:
    batch_files = sorted(CACHE_DIR.glob("batch_*.pkl"))
    scaler = StandardScaler()
    total_items_extracted = 0

    # --- [Step 1: Extraction & Fit Scaler] ---
    for batch_file in tqdm(batch_files, desc="Extraction & Fitting"):
        batch_id = batch_file.stem
        with open(batch_file, "rb") as f: batch_data = pickle.load(f)
        batch_raw_pool = []
        for item in tqdm(batch_data['segments'], desc=f"  [{batch_id}]", leave=False):
            # Extract 45D HI
            # if cyc >= 148 and cyc <= 150 and item['cell'] == "b1c0":
            #     item.to_csv(f"./{cyc}_b1c0_segment.csv", index=False)
            #     import pdb; pdb.set_trace()
            hi = extract_75d_hi(item['df'], "C" if item['mode_label'] == 1 else "D")
            batch_raw_pool.append({
                "cell": item['cell'], "cyc": item['cyc'], 
                "x": hi, # Use 'x' for raw features per convention
                "soc_label": item['soc_label'], "mode_label": item['mode_label'],
                "length_p": item['length_p'], "y": item.get('rul', item.get('y', 0.0)),
                "capacity": item.get('capacity', 0.0)
            })
        # Partial fit for memory efficiency
        raw_x_batch = [item['x'] for item in batch_raw_pool]
        scaler.partial_fit(raw_x_batch)
        
        # Save intermediate raw features
        with open(FEATURE_BATCH_DIR / f"raw_{batch_id}.pkl", "wb") as f: pickle.dump(batch_raw_pool, f)
        total_items_extracted += len(batch_raw_pool)
        del batch_data, batch_raw_pool; gc.collect()

    # --- [Step 2: Scaling & Masking] ---
    batch_feature_pool = []
    raw_batch_files = sorted(FEATURE_BATCH_DIR.glob("raw_batch_*.pkl"))
    for raw_file in tqdm(raw_batch_files, desc="Scaling & Saving"):
        batch_id = raw_file.stem.replace("raw_", "")
        with open(raw_file, "rb") as f: batch_pool = pickle.load(f)
        
        raw_x_vals = [item['x'] for item in batch_pool]
        scaled_x_vals = scaler.transform(raw_x_vals)
        
        for i, item in enumerate(batch_pool):
            s_x = scaled_x_vals[i]
            # Scenario-aware Masking (f0-14: Common, f15-29: Discharge, f30-44: Charge)
            if item['mode_label'] == 1: s_x[15:30] = 0.0 # Charge mode: mask discharge features
            else: s_x[30:45] = 0.0 # Discharge mode: mask charge features
            
            # Inject Metadata (SOC, Mode, Length)
            meta = np.array([item['soc_label'], item['mode_label'], item['length_p']], dtype=np.float32)
            item['scaled_x'] = np.concatenate([s_x, meta]) # Use 'scaled_x' for scaled features
            
        # Save final batch
        with open(FEATURE_BATCH_DIR / f"final_{batch_id}.pkl", "wb") as f: pickle.dump(batch_pool, f)
        batch_feature_pool.extend(batch_pool)
        raw_file.unlink() # Cleanup raw cache
        
    with open(SCALER_CACHE_PATH, "wb") as f: pickle.dump(scaler, f)
    print(f"[Success] Extraction and Scaling complete. Total items: {len(batch_feature_pool):,}")


### 3.1.1 45 HIs analysis for 6 scenarios

In [ ]:


# 가독성을 위한 라벨 매핑
soc_map = {
    -2: 'H(100-70%)',
    -1: 'M(70-30%)',
     0: 'L(30-0%)'
}

mode_map = {
    1: 'Charge',
    0: 'Discharge'
}

#### original_ver

In [ ]:
# --- 1. 데이터 로드 및 통합 DataFrame 생성 ---
raw_batch_files = sorted(FEATURE_BATCH_DIR.glob("*.pkl"))

all_data = []
plot_cells = []

if DATASET_TYPE == "mit":
    plot_cells = mit_cell_ids  # 분석 및 시각화에 사용할 셀 샘플
elif DATASET_TYPE == "hust":
    plot_cells = hust_cell_ids  # 분석 및 시각화에 사용할 셀 샘플

# 분석의 정확도를 위해 최소 3개 이상의 배치를 샘플링 권장
for f in tqdm(raw_batch_files, desc="Loading batches for analysis"):
    with open(f, "rb") as fin:
        all_data.extend(pickle.load(fin))

# 45D HI 추출
hi_features = [item['x'] for item in all_data]

# 메타데이터 추출
meta_info = [
    {
        'cell': item['cell'],
        'cyc': item['cyc'],
        'soc_label': item['soc_label'],
        'mode_label': item['mode_label'],
        'capacity': item['capacity']
    }
    for item in all_data
]

# 피처 DataFrame에 컬럼명(hi_feature_names) 할당
df_features = pd.DataFrame(hi_features)

if 'hi_feature_names' in globals() and len(hi_feature_names) == df_features.shape[1]:
    df_features.columns = hi_feature_names
else:
    df_features.columns = [f"Feature_{i}" for i in range(df_features.shape[1])]

# 통합 DataFrame 생성
df_main = pd.concat([pd.DataFrame(meta_info), df_features], axis=1)

# 가독성을 위한 라벨 매핑
soc_map = {-2: 'H(100-70%)', -1: 'M(70-30%)', 0: 'L(30-0%)'}
mode_map = {1: 'Charge', 0: 'Discharge'}

df_main['SOC'] = df_main['soc_label'].map(soc_map)
df_main['Mode'] = df_main['mode_label'].map(mode_map)
df_main['Scenario'] = df_main['Mode'] + "_" + df_main['SOC']

# =====================================================================
# 모든 45개 피처 및 시나리오가 포함된 최종 DataFrame 저장
# =====================================================================
output_csv_path = Path(f"./{DATASET_TYPE}_all_segments_45_features.csv")

# 저장하기 전에 정렬 (셀 -> 사이클 -> 모드 -> SOC 순)
df_main_sorted = df_main.sort_values(
    by=['cell', 'cyc', 'mode_label', 'soc_label']
)

df_main_sorted.to_csv(output_csv_path, index=False)

print(
    f">> [Export Complete] 45개 전체 피처가 포함된 데이터가 저장되었습니다: "
    f"{output_csv_path}"
)
print(
    f">> Total Segments: {len(df_main_sorted)}, "
    f"Total Columns: {len(df_main_sorted.columns)}\n"
)

# =====================================================================

# --- 2. 시각화 (시간 순서에 따른 개별 HI 변화) ---
for target_feat in hi_feature_names:

    for cell in plot_cells:

        df_cell_only = df_main[df_main['cell'] == cell]

        if df_cell_only.empty:
            continue

        plt.figure(figsize=(15, 7))
        target_cell = cell

        for scen in df_cell_only['Scenario'].unique():

            subset = df_cell_only[
                df_cell_only['Scenario'] == scen
            ].sort_values('cyc')

            plt.scatter(
                subset['cyc'],
                subset[target_feat],
                s=20,
                alpha=0.7,
                label=scen
            )

        plt.xlabel("Cycle Number")
        plt.ylabel(f"Raw HI Value ({target_feat})")
        plt.title(
            f"Temporal Change of Feature {target_feat} "
            f"(Cell: {target_cell})"
        )
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.5)

        save_dir = Path(FIGURE_ROOT) / cell
        save_dir.mkdir(parents=True, exist_ok=True)

        plt.savefig(
            save_dir / f"{DATASET_TYPE}_{target_feat}_temporal_change.png",
            dpi=120
        )
        plt.close()  # 메모리 누수 방지

#### new version

In [ ]:
# import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt
# import numpy as np
# import pickle
# import os
# from pathlib import Path
# from tqdm.auto import tqdm

# # --- 1. 사용할 피처 이름 정의 ---


# # --- 2. 데이터 로드 및 통합 DataFrame 생성 ---
# raw_batch_files = sorted(FEATURE_BATCH_DIR.glob("*.pkl"))
# all_data = []

# for f in tqdm(raw_batch_files, desc="Loading batches for analysis"):
#     with open(f, "rb") as fin:
#         all_data.extend(pickle.load(fin))

# # 특징을 DataFrame으로 변환하며 컬럼명 할당
# df_features = pd.DataFrame([item['x'] for item in all_data])

# if len(df_features.columns) == 45:
#     df_features.columns = hi_feature_names
# else:
#     print(
#         f"⚠️ Warning: 피처 배열 길이({len(df_features.columns)})가 "
#         f"hi_feature_names(45)와 다릅니다!"
#     )
#     hi_feature_names = df_features.columns.tolist()

# # 메타데이터 결합
# meta_info = [
#     {
#         'cell': item['cell'],
#         'cyc': item['cyc'],
#         'soc_label': item['soc_label'],
#         'mode_label': item['mode_label'],
#         'capacity': item['capacity']
#     }
#     for item in all_data
# ]

# df_main = pd.concat(
#     [pd.DataFrame(meta_info), df_features],
#     axis=1
# )

# df_main['SOC'] = df_main['soc_label'].map(soc_map)
# df_main['Mode'] = df_main['mode_label'].map(mode_map)
# df_main['Scenario'] = df_main['Mode'] + "_" + df_main['SOC']

# # --- 3. 시각화 대상 셀 선정 ---
# available_cells = df_main['cell'].unique()

# print(f"데이터셋에 존재하는 전체 셀 개수: {len(available_cells)}개")

# # (옵션 A) 특정 셀 수동 지정
# plot_cells = [
#     c for c in
#     ['b1c11', 'b2c7', 'b3c46', 'b1c1', 'b2c6', 'b3c38']
#     if c in available_cells
# ]

# print(f"플롯 대상 셀: {plot_cells}")

# # --- 4. 45개 피처 시각화 및 저장 ---
# for cell in tqdm(plot_cells, desc="Processing Cells"):

#     df_cell = df_main[df_main['cell'] == cell]

#     cell_dir = FIGURE_ROOT / cell
#     os.makedirs(cell_dir, exist_ok=True)

#     for i, target_feat in enumerate(
#         tqdm(
#             hi_feature_names,
#             desc=f"Features for {cell}",
#             leave=False
#         )
#     ):

#         plt.figure(figsize=(15, 7))

#         # 시나리오별 scatter plot
#         for scen in df_cell['Scenario'].unique():
#             subset = (
#                 df_cell[df_cell['Scenario'] == scen]
#                 .sort_values('cyc')
#             )

#             plt.scatter(
#                 subset['cyc'],
#                 subset[target_feat],
#                 s=20,
#                 alpha=0.7,
#                 label=scen
#             )

#         plt.xlabel("Cycle Number")
#         plt.ylabel(f"Raw HI Value ({target_feat})")

#         plt.title(
#             f"Temporal Change of {target_feat} "
#             f"(Cell: {cell})"
#         )

#         plt.legend(
#             bbox_to_anchor=(1.05, 1),
#             loc='upper left'
#         )

#         plt.grid(
#             True,
#             linestyle='--',
#             alpha=0.5
#         )

#         # 파일명 저장
#         save_name = (
#             f"{DATASET_TYPE}_feature_"
#             f"{str(i + 1).zfill(2)}_"
#             f"{target_feat}_temporal_change.png"
#         )

#         save_path = cell_dir / save_name

#         plt.savefig(
#             save_path,
#             bbox_inches='tight'
#         )

#         # 메모리 보호
#         plt.close()

# print("✅ 45개 피처 플롯 저장이 모두 완료되었습니다!")

## Phase 3.2: Feature & Label Analysis

In [ ]:
def plot_phase4_feature_analysis(pool):
    import random
    if not pool: return
    
    # Decide which feature to use
    sample_item = pool[0]
    feature_key = 'scaled_x' if 'scaled_x' in sample_item else 'x'
    print(f"[Analysis] Using feature key: '{feature_key}'")
    
    num_features = len(sample_item[feature_key])
    sample_pool = random.sample(pool, min(5000, len(pool)))
    df_features = pd.DataFrame([{
        "cyc": s["cyc"], "y": s.get("y", s.get("rul", 0)), "cap": s["capacity"],
        **{f"f{i}": val for i, val in enumerate(s[feature_key])}
    } for s in sample_pool])
    
    fig, axes = plt.subplots(2, 2, figsize=(24, 20))
    corr = df_features[[f"f{i}" for i in range(num_features)]].corr()
    sns.heatmap(corr, annot=False, cmap='coolwarm', ax=axes[0,0])
    axes[0,0].set_title(f"Correlation Matrix (All {num_features} Features)")
    sns.scatterplot(data=df_features, x='cap', y='f0', alpha=0.3, ax=axes[0,1])
    axes[0,1].set_title(f"SOH (Cap) vs {feature_key}[0]")
    sns.lineplot(data=df_features, x='cyc', y='y', ax=axes[1,0])
    axes[1,0].set_title("RUL Label vs Cycle Number")
    sns.boxplot(data=df_features.melt(value_vars=[f"f{i}" for i in range(num_features)]), x='variable', y='value', ax=axes[1,1])
    axes[1,1].set_title(f"Feature Distributions ({feature_key})")
    axes[1,1].tick_params(axis='x', rotation=90)
    plt.tight_layout()
    plt.savefig(FIGURE_ROOT / f"{DATASET_TYPE}_feature_analysis.png", dpi=300, bbox_inches='tight')
    plt.show()

if 'batch_feature_pool' in globals() and len(batch_feature_pool) > 0:
    plot_phase4_feature_analysis(batch_feature_pool)
else:
    print("No data found in memory for analysis.")


## Phase 4: Final Saving

In [ ]:
if START_PHASE <= 4 or not FINAL_TENSOR_PATH.exists():
    print(f"[Phase 4] Starting Final Tensor Saving for {DATASET_TYPE}...")
    if 'batch_feature_pool' not in globals() or not batch_feature_pool:
        print("[Error] No feature batches found in memory. Please run Phase 3 first.")
    else:
        final_data = []
        for item in tqdm(batch_feature_pool, desc="Formatting Optimized Tensors"):
            # Use 'scaled_x' if available, otherwise 'x'
            x_val = item.get('scaled_x', item.get('x'))
            
            final_data.append({
                "cell": item['cell'], "cyc": item['cyc'], "x": x_val,
                "soc_label": item['soc_label'], "mode_label": item['mode_label'],
                "length_p": item['length_p'], 
                "y": item.get('y', item.get('rul', 0.0)),
                "capacity": item.get('capacity', 0.0)
            })
        with open(FINAL_TENSOR_PATH, "wb") as f: 
            pickle.dump(final_data, f)
        print(f"[Success] Mapped and saved {len(final_data):,} samples to {FINAL_TENSOR_PATH}")
else:
    print(f"[Phase 4] Skipping Final Tensor Saving. Optimized file already exists.")


### Phase 4.1 : check for correlation with HIs and capacity

In [ ]:
import re
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

def validate_hi_theory_comprehensive(pool, hi_feature_names, all_cells=None, figure_root=None):
    if not pool:
        print("분석할 데이터(pool)가 없습니다.")
        return

    # 0. 시각화 저장 경로 설정
    hi_analysis_dir = None
    if figure_root:
        hi_analysis_dir = Path(figure_root) / "HI_analysis" / DATASET_TYPE
        hi_analysis_dir.mkdir(parents=True, exist_ok=True)
        print(f">> 시각화 결과가 {hi_analysis_dir} 폴더에 저장됩니다.")

    # 1. 시나리오별 이론적 예상 방향 정의 (guidelines/HI_analysis_hust.md 45개 피처 반영)
    def get_common_expectations():
        return {
            # --- 1. 공통 및 통계 지표 (15개) ---
            "c1_1_mean_v": {"C-H": -1, "C-M": -1, "C-L": -1, "D-H": 1, "D-M": 1, "D-L": 1},
            "c1_2_var_v":  {"C-H": -1, "C-M": -1, "C-L": -1, "D-H": -1, "D-M": -1, "D-L": -1},
            "c1_3_skew_v": {"C-H": -1, "C-M": -1, "C-L": -1, "D-H": -1, "D-M": -1, "D-L": -1},
            "c1_4_kurt_v": {"C-H": -1, "C-M": -1, "C-L": -1, "D-H": -1, "D-M": -1, "D-L": -1},
            "c1_5_mean_t": {"C-H": -1, "C-M": -1, "C-L": -1, "D-H": -1, "D-M": -1, "D-L": -1},
            "c1_6_delta_t": {"C-H": -1, "C-M": -1, "C-L": -1, "D-H": -1, "D-M": -1, "D-L": -1},
            "c1_7_dtdt":   {"C-H": -1, "C-M": -1, "C-L": -1, "D-H": -1, "D-M": -1, "D-L": -1},
            "c1_8_mean_i": {"C-H": 1, "C-M": 1, "C-L": 1},  # 방전은 CC이므로 매핑 생략(N/A)
            "c1_9_var_i":  {"C-H": -1, "C-M": -1, "C-L": -1},  # 방전은 N/A
            "c1_10_ent_v": {"C-H": -1, "C-M": -1, "C-L": -1, "D-H": -1, "D-M": -1, "D-L": -1},
            "c1_11_ent_t": {"C-H": -1, "C-M": -1, "C-L": -1, "D-H": -1, "D-M": -1, "D-L": -1},
            "c1_12_corr_vi": {"C-H": 1, "C-M": 1, "C-L": 1, "D-H": 1, "D-M": 1, "D-L": 1},
            "c1_13_corr_vt": {"C-H": 1, "C-M": 1, "C-L": 1, "D-H": 1, "D-M": 1, "D-L": 1},
            "c1_14_de_dq": {"C-H": 1, "C-M": 1, "C-L": 1, "D-H": 1, "D-M": 1, "D-L": 1},
            "c1_15_power_var": {"C-H": -1, "C-M": -1, "C-L": -1, "D-H": -1, "D-M": -1, "D-L": -1},

            # --- 2. 방전 특화 지표 (15개) ---
            # 충전 시나리오는 매핑 안 함으로써 N/A 처리
            "c2_dis_1_time": {"D-H": 1, "D-M": 1, "D-L": 1},
            "c2_dis_2_capacity": {"D-H": 1, "D-M": 1, "D-L": 1},
            "c2_dis_3_ir_drop": {"D-H": -1},  # 초기 전압강하 (D-M, D-L 제외)
            "c2_dis_4_dv_dt": {"D-H": -1, "D-M": -1, "D-L": -1},
            "c2_dis_5_ic_peak": {"D-H": 1, "D-M": 1, "D-L": 1},
            "c2_dis_6_max_temp": {"D-H": -1, "D-M": -1, "D-L": -1},
            "c2_dis_7_cc_time": {"D-H": 1, "D-M": 1, "D-L": 1},
            "c2_dis_8_v_integral": {"D-H": 1, "D-M": 1, "D-L": 1},
            "c2_dis_9_v_var": {"D-H": -1, "D-M": -1, "D-L": -1},
            "c2_dis_10_v_skew": {"D-H": -1, "D-M": -1, "D-L": -1},
            "c2_dis_11_segment_q": {"D-H": 1, "D-M": 1, "D-L": 1},
            "c2_dis_12_end_drop_rt": {"D-L": -1},  # 방전 말기에만 해당
            "c2_dis_13_energy_loss": {"D-H": -1, "D-M": -1, "D-L": -1},
            "c2_dis_14_efficiency": {"D-H": 1, "D-M": 1, "D-L": 1},
            "c2_dis_15_dcir": {"D-H": -1, "D-M": -1, "D-L": -1},

            # --- 3. 충전 특화 지표 (15개) ---
            # 방전 시나리오는 매핑 안 함으로써 N/A 처리
            "c3_cha_1_time": {"C-H": 1, "C-M": 1, "C-L": 1},
            "c3_cha_2_cc_capacity": {"C-H": 1, "C-M": 1, "C-L": 1},
            "c3_cha_3_polarization": {"C-L": -1},  # 충전 개시 분극 (C-H, C-M 제외)
            "c3_cha_4_dv_dt": {"C-H": -1, "C-M": -1, "C-L": -1},
            "c3_cha_5_ic_peak": {"C-H": 1, "C-M": 1, "C-L": 1},
            "c3_cha_6_cv_time": {"C-H": -1, "C-M": -1, "C-L": -1},
            "c3_cha_7_total_q": {"C-H": 1, "C-M": 1, "C-L": 1},
            "c3_cha_8_temp_rise": {"C-H": -1, "C-M": -1, "C-L": -1},
            "c3_cha_9_v_var": {"C-H": -1, "C-M": -1, "C-L": -1},
            "c3_cha_10_efficiency": {"C-H": 1, "C-M": 1, "C-L": 1},
            "c3_cha_11_seg_time": {"C-H": 1, "C-M": 1, "C-L": 1},
            "c3_cha_12_cv_i_drop": {"C-H": -1, "C-M": -1, "C-L": -1},
            "c3_cha_13_cv_q": {"C-H": 1, "C-M": 1, "C-L": 1},
            "c3_cha_14_v_max_spd": {"C-H": -1, "C-M": -1, "C-L": -1},
            "c3_cha_15_dcir": {"C-H": -1, "C-M": -1, "C-L": -1},
        }

    raw_expectations = get_common_expectations()
    expectations = {s: {} for s in ["C-H", "C-M", "C-L", "D-H", "D-M", "D-L"]}

    for feat, scen_dict in raw_expectations.items():
        for scen, direct in scen_dict.items():
            if scen in expectations:
                expectations[scen][feat] = direct

    feature_key = 'scaled_x' if 'scaled_x' in pool[0] else 'x'

    # 2. DataFrame 변환
    data_rows = []

    for item in tqdm(pool, desc="Preparing Data for Validation"):
        mode_prefix = "C" if item.get("mode_label", 1) == 1 else "D"
        soc_val = item.get("soc_label", 0)

        soc_suffix = (
            "H" if soc_val == -2
            else "M" if soc_val == -1
            else "L"
        )

        scenario_name = f"{mode_prefix}-{soc_suffix}"

        row = {
            "cell_id": item.get("cell", "unknown"),
            "capacity": item["capacity"],
            "scenario": scenario_name
        }

        for i, val in enumerate(item[feature_key]):
            if i < len(hi_feature_names):
                row[hi_feature_names[i]] = val

        data_rows.append(row)

    df = pd.DataFrame(data_rows)
    
    # [추가됨] 특정 셀 단위의 시각화 함수
    def plot_cell_specific_correlations(df_main, target_cell_ids):
        """
        지정된 셀 ID 리스트(target_cell_ids)에 대해 개별 셀의 상관관계 그래프를 생성합니다.
        셀 내부에서도 충전(Charge)과 방전(Discharge) 모드가 혼재할 수 있으므로,
        이를 분리하여 시나리오별 기댓값을 정확히 적용합니다.
        """
        if not hi_analysis_dir or df_main.empty or not target_cell_ids:
            return

        print(f"\n>> 지정된 셀({len(target_cell_ids)}개)에 대한 개별 시각화 시작...")

        for cell_id in target_cell_ids:
            df_cell = df_main[df_main['cell_id'] == cell_id]

            if df_cell.empty:
                print(f"  - Cell {cell_id}: 데이터가 없어 건너뜁니다.")
                continue

            # 셀 하나라도 내부에는 충전/방전 및 SOC 구간별 세그먼트가 섞여 있음
            # 따라서 셀 내에서도 시나리오(scenario)별로 분리하여 기댓값을 적용해야 함
            cell_scenarios = df_cell['scenario'].unique()

            for scen in cell_scenarios:
                df_cell_scen = df_cell[df_cell['scenario'] == scen]

                # 최소 샘플 수 확인 (너무 적으면 상관관계 계산이 무의미함)
                if len(df_cell_scen) < 10:
                    continue

                title = f"Cell: {cell_id} | Scenario: {scen}"
                filename = f"cell_{cell_id}_scen_{scen}_corr.png"

                # 기존 시나리오 기반 plot 헬퍼 함수 재사용
                plot_correlations(
                    df_sub=df_cell_scen,
                    title=title,
                    filename=filename,
                    current_scen=scen
                )

        print(">> 개별 셀 시각화 완료.\n")

    # 시각화 헬퍼 함수
    def plot_correlations(df_sub, title, filename, current_scen=None):
        if not hi_analysis_dir or df_sub.empty:
            return

        scen_expect = expectations.get(current_scen, {})

        corrs = []
        colors = []

        for feat in hi_feature_names:
            if feat in df_sub.columns:
                c = df_sub['capacity'].corr(df_sub[feat])
                corrs.append(c if not pd.isna(c) else 0)
                
                if pd.isna(c) or feat not in scen_expect:
                    colors.append('grey')
                else:
                    expected = scen_expect[feat]

                    if (expected > 0 and c > 0) or (expected < 0 and c < 0):
                        colors.append('blue')
                    else:
                        colors.append('red')
            else:
                corrs.append(0)
                colors.append('grey')

        plt.figure(figsize=(15, 6))
        sns.barplot(x=hi_feature_names, y=corrs, palette=colors)
        plt.xticks(rotation=90, fontsize=8)
        plt.axhline(0, color='black', linewidth=1)
        plt.ylim(-1.1, 1.1)

        plt.title(
            f"{title}\n(Blue: Match, Red: Contradict, Grey: N/A)",
            fontsize=14
        )
        plt.ylabel("Corr with Capacity")
        plt.tight_layout()

        plt.savefig(hi_analysis_dir / filename, dpi=120)
        plt.close()

    # 시각화 실행
    scenarios = sorted(df['scenario'].unique())

    for scen in scenarios:
        df_scen = df[df['scenario'] == scen]

        if len(df_scen) >= 20:
            plot_correlations(
                df_scen,
                f"Scenario: {scen}",
                f"scen_{scen}_corr.png",
                current_scen=scen
            )
        
    # [추가됨] 원하는 타겟 셀 ID를 지정하여 셀 단위 시각화 호출
    # (예시: HUST 데이터셋의 1-1, 1-2번 셀)
    if DATASET_TYPE == "hust":
        target_cells_to_plot = ['1-1', '1-2', '3-1', '5-1'] # 분석을 원하는 셀 ID 입력
    elif DATASET_TYPE == "mit":
        target_cells_to_plot = ['b1c0', 'b1c11', 'b2c0', 'b2c47', 'b3c0','b3c38'] # MIT 데이터셋의 예시 셀 ID 입력
    
    plot_cell_specific_correlations(df, target_cells_to_plot)

    # 요약 출력
    print("\n" + "=" * 95)
    print(f"{'[HI Validation Summary by 6-SCENARIOS]':^95}")
    print("-" * 95)
    print(
        f"{'Scenario':<12} | {'Match %':<10} | {'Perfect(P)':<12} | "
        f"{'Weak(W)':<10} | {'Contradict(C)':<12} | {'N'}"
    )
    print("-" * 95)

    for scen in scenarios:
        df_scen = df[df['scenario'] == scen]

        if len(df_scen) < 20:
            continue

        scen_expect = expectations.get(scen, {})
        res = {"P": 0, "W": 0, "C": 0, "total": 0}

        for feat in hi_feature_names:
            if feat in df_scen.columns and feat in scen_expect:
                res["total"] += 1

                c = df_scen['capacity'].corr(df_scen[feat])

                if pd.isna(c):
                    continue

                expected = scen_expect[feat]

                if (expected > 0 and c > 0) or (expected < 0 and c < 0):
                    if abs(c) >= 0.3:
                        res["P"] += 1
                    else:
                        res["W"] += 1
                else:
                    res["C"] += 1

        if res["total"] > 0:
            acc = (res["P"] / res["total"]) * 100

            print(
                f"{scen:<12} | {acc:>8.1f}% | "
                f"{res['P']:>10} | {res['W']:>8} | "
                f"{res['C']:>12} | {len(df_scen)}"
            )

    print("=" * 95)


# =====================================================================
# 메인 실행부: preprocess.ipynb에서 저장한 파일 로드 및 실행
# =====================================================================
if __name__ == "__main__":
    # 1. 파일 경로 설정 (DATASET_TYPE에 따라 자동 선택)
    processed_path = FINAL_TENSOR_PATH

    if processed_path.exists():
        print(f">> 전처리된 배치 파일을 로드합니다: {processed_path}")

        with open(processed_path, "rb") as f:
            batch_feature_pool = pickle.load(f)

        # 2. 피처 이름 정의
        # if "hi_feature_names" not in globals():
        #     hi_feature_names = [
        #         "mean_v", "var_v", "mean_t", "delta_t",
        #         "dtdt", "mean_i", "var_i", "ent_v", "de_dq",
        #         "time", "capacity", "ir_drop", "dv_dt",
        #         "max_temp", "v_integral", "dcir",
        #         "cc_capacity", "polarization", "cv_time",
        #         "total_q", "seg_time", "cv_i_drop"
        #     ]

        # 3. 함수 실행
        validate_hi_theory_comprehensive(
            batch_feature_pool,
            hi_feature_names,
            all_cells=all_cells if "all_cells" in globals() else None,
            figure_root=FIGURE_ROOT if "FIGURE_ROOT" in globals()
            else "./outputs/figures"
        )

    else:
        print(f"!! 파일을 찾을 수 없습니다: {processed_path}")

## Phase 5: Visualization

In [ ]:
from sklearn.decomposition import PCA
def analyze_final_hi_dataset(dataset_type):
    FEATURE_BATCH_DIR = PROCESSED_DATA_ROOT / f"{dataset_type}_feature_batches"
    final_batch_files = sorted(FEATURE_BATCH_DIR.glob("final_batch_*.pkl"))
    
    if not final_batch_files:
        print(f"Error: No batch files found in {FEATURE_BATCH_DIR}")
        return

    all_data = []
    print(f"Loading {len(final_batch_files)} batches for visualization...")
    for bf in tqdm(final_batch_files, desc="Loading batches"): 
        with open(bf, "rb") as f: 
            batch_data = pickle.load(f)
            # Ensure consistent key naming ('scaled_x' -> 'x')
            for item in batch_data:
                if 'scaled_x' in item and 'x' not in item:
                    item['x'] = item['scaled_x']
                if 'rul' in item and 'y' not in item:
                    item['y'] = item['rul']
            all_data.extend(batch_data)
    
    if not all_data:
        print("Error: Loaded data pool is empty.")
        return

    df = pd.DataFrame(all_data)
    x_expanded = pd.DataFrame(df['x'].tolist())
    x_expanded.columns = [f"HI_{i}" for i in range(x_expanded.shape[1])]
    df = pd.concat([df.drop('x', axis=1), x_expanded], axis=1)
    
    df['batch_id'] = df['cell'].apply(lambda cid: get_batch_id(cid, dataset_type))
    df['scen_str'] = df['soc_label'].map({-2: 'High SOC', -1: 'Mid SOC', 0: 'Low SOC'})
    df['mode_str'] = df['mode_label'].map({1: 'Charge', 0: 'Discharge'})
    
    fig = plt.figure(figsize=(22, 18))
    ax1 = plt.subplot(2, 2, 1)
    # Only plot first 5 features for clarity
    sns.boxplot(data=df.melt(id_vars=['scen_str'], value_vars=[f"HI_{i}" for i in range(5)]), x='variable', y='value', hue='scen_str', ax=ax1)
    ax1.set_title("Feature Distribution by SOC Scenario (Top 5 HIs)")
    
    ax2 = plt.subplot(2, 2, 2)
    sns.violinplot(data=df, x='batch_id', y='HI_0', hue='mode_str', split=True, ax=ax2)
    ax2.set_title("HI_0 (Mean Voltage) Distribution by Batch and Mode")
    
    ax3 = plt.subplot(2, 2, 3)
    pca = PCA(n_components=2); hi_cols = [f"HI_{i}" for i in range(x_expanded.shape[1])]
    pca_res = pca.fit_transform(df[hi_cols].dropna())
    sc = plt.scatter(pca_res[:, 0], pca_res[:, 1], c=df.dropna(subset=hi_cols)['y'], cmap='viridis_r', s=10)
    plt.colorbar(sc, label='RUL')
    ax3.set_title("PCA Analysis: 2D Projection colored by RUL")
    
    plt.tight_layout()
    plt.savefig(figure_path / f"{dataset_type}_hi_analysis.png", dpi=300, bbox_inches='tight')
    plt.show()

def plot_hi_comprehensive_analysis(df):
    """Creates a comprehensive dashboard of HI characteristics."""
    import matplotlib.gridspec as gridspec
    hi_cols = [f"HI_{i}" for i in range(10)] # Top 10 HIs
    
    fig = plt.figure(figsize=(24, 16))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    
    # 1. HI Trend over Cycles (Top 3 HIs)
    ax1 = fig.add_subplot(gs[0, :2])
    for i in range(3):
        sns.lineplot(data=df, x='cyc', y=f'HI_{i}', label=f'HI_{i}', ax=ax1, alpha=0.7)
    ax1.set_title("Degradation Trends: Top 3 HIs over Cycle Number")
    ax1.set_xlabel("Cycle Number")
    ax1.set_ylabel("Scaled Value")
    
    # 2. HI vs Capacity Correlation Heatmap
    ax2 = fig.add_subplot(gs[0, 2])
    corr_target = df[hi_cols + ['capacity']].corr()[['capacity']].sort_values(by='capacity', ascending=False)
    sns.heatmap(corr_target, annot=True, cmap='RdBu_r', ax=ax2)
    ax2.set_title("HI Correlation with Capacity (SOH)")
    
    # 3. HI Distribution across SOC Scenarios (HI_0)
    ax3 = fig.add_subplot(gs[1, 0])
    sns.boxplot(data=df, x='scen_str', y='HI_0', hue='mode_str', ax=ax3)
    ax3.set_title("HI_0 Distribution by SOC & Mode")
    
    # 4. Feature Space Clustering (PCA)
    ax4 = fig.add_subplot(gs[1, 1:])
    pca = PCA(n_components=2)
    pca_res = pca.fit_transform(df[hi_cols].dropna())
    scatter = ax4.scatter(pca_res[:, 0], pca_res[:, 1], c=df.dropna(subset=hi_cols)['capacity'], 
                         cmap='magma', s=5, alpha=0.5)
    plt.colorbar(scatter, ax=ax4, label='Capacity (Ah)')
    ax4.set_title("PCA Projection: HI Feature Space colored by SOH")
    
    # 5. Density of SOH Labels
    ax5 = fig.add_subplot(gs[2, 0])
    sns.kdeplot(data=df, x='capacity', hue='scen_str', fill=True, ax=ax5)
    ax5.set_title("Capacity Density by Scenario")
    
    # 6. HI_1 vs HI_2 scatter colored by RUL
    ax6 = fig.add_subplot(gs[2, 1:])
    scatter2 = ax6.scatter(df['HI_1'], df['HI_2'], c=df['y'], cmap='viridis_r', s=5, alpha=0.4)
    plt.colorbar(scatter2, ax=ax6, label='RUL')
    ax6.set_title("HI Interaction: HI_1 vs HI_2 colored by RUL")
    
    plt.tight_layout()
    fig.savefig(figure_path / f"{DATASET_TYPE}_hi_comprehensive_analysis.png", dpi=300)
    plt.show()

# Execute Comprehensive Analysis
if 'all_data' in locals() and len(all_data) > 0:
    # Re-use the already loaded and processed DataFrame if available
    df_final = pd.DataFrame(all_data)
    x_expanded = pd.DataFrame(df_final['x'].tolist())
    x_expanded.columns = [f"HI_{i}" for i in range(x_expanded.shape[1])]
    df_final = pd.concat([df_final.drop('x', axis=1), x_expanded], axis=1)
    df_final['scen_str'] = df_final['soc_label'].map({-2: 'High SOC', -1: 'Mid SOC', 0: 'Low SOC'})
    df_final['mode_str'] = df_final['mode_label'].map({1: 'Charge', 0: 'Discharge'})
    
    plot_hi_comprehensive_analysis(df_final)

analyze_final_hi_dataset(DATASET_TYPE)

In [ ]:
def plot_comprehensive_hi_analysis(pool, save_path=f"{figure_path}/{DATASET_TYPE}_comprehensive_HI_analysis.png"):
    if not pool:
        print("No data in pool for analysis.")
        return

    # --- [디자인 테마 및 폰트 전역 설정] ---
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams.update({
        'font.size': 12,
        'axes.titlesize': 16,
        'axes.labelsize': 13,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'legend.fontsize': 11,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white'
    })

    # 1. 데이터 프레임 준비
    num_features = len(pool[0]["scaled_x"])
    sample_size = min(10000, len(pool))
    sample_pool = random.sample(pool, sample_size)

    data_dict = []
    for s in sample_pool:
        row = {
            "cell": s["cell"],
            "cyc": s["cyc"],
            "y_rul": s["y"],
            "cap_soh": s["capacity"]
        }
        for i, val in enumerate(s["scaled_x"]):
            row[f"f{i}"] = val
        data_dict.append(row)

    df = pd.DataFrame(data_dict)
    feature_cols = [f"f{i}" for i in range(num_features)]

    fig, axes = plt.subplots(2, 2, figsize=(22, 18))
    plt.subplots_adjust(hspace=0.35, wspace=0.25)


    print(f"DataFrame for Analysis: {df.shape[0]} samples, {df.shape[1]} columns (including features)")
    print(f"Feature columns: {feature_cols}")

    # --- [Plot 1: HI vs HI Correlation Matrix (45D)] ---
    corr_matrix = df[feature_cols].corr()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix,
        mask=mask,
        cmap='coolwarm',
        vmin=-1,
        vmax=1,
        center=0,
        ax=axes[0, 0],
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8, 'label': 'Correlation Coefficient'}
    )
    axes[0, 0].set_title(f"A. HI vs HI Correlation Matrix ({num_features}D)", fontweight='bold', pad=15)

    # --- [Plot 2: Top 20 Feat. Correlation with Target Variables] ---
    corrs_rul = df[feature_cols].corrwith(df["y_rul"]).abs()
    corrs_cap = df[feature_cols].corrwith(df["cap_soh"]).abs()

    top_20_idx = (corrs_rul + corrs_cap).sort_values(ascending=False).head(20).index
    df_top_corr = pd.DataFrame({
        "Feature": top_20_idx,
        "RUL (Corr)": corrs_rul[top_20_idx],
        "Capacity (Corr)": corrs_cap[top_20_idx]
    }).melt(id_vars="Feature", var_name="Target", value_name="Abs Corr")

    sns.barplot(
        data=df_top_corr,
        x="Abs Corr",
        y="Feature",
        hue="Target",
        palette={"RUL (Corr)": "#1f77b4", "Capacity (Corr)": "#ff7f0e"},
        ax=axes[0, 1],
        edgecolor='black',
        linewidth=0.7
    )
    axes[0, 1].set_title("B. Top 20 Features: Absolute Correlation", fontweight='bold', pad=15)
    axes[0, 1].legend(loc='lower right', frameon=True, shadow=True)
    sns.despine(ax=axes[0, 1])

    # --- [Plot 3: Feature Profile for Single Segment (Stem Plot 스타일)] ---
    sample_seg = random.choice(pool)
    cid, cyc, mode = sample_seg['cell'], sample_seg['cyc'], sample_seg['mode_label']
    feat_values = np.array(sample_seg["scaled_x"])

    axes[1, 0].axhline(0, color='black', linewidth=1.2, alpha=0.7)

    groups = [
        (range(0, 15), feat_values[0:15], '#2ca02c', 'o', 'C1'),
        (range(15, 30), feat_values[15:30], '#d62728', 's', 'C2'),
        (range(30, 45), feat_values[30:45], '#9467bd', '^', 'C3')
    ]

    for idx_range, values, color, marker, label in groups:
        axes[1, 0].vlines(idx_range, ymin=0, ymax=values, color=color, alpha=0.5, linewidth=1.5)
        axes[1, 0].scatter(idx_range, values, color=color, marker=marker, s=60,
                           label=label, edgecolors='black', zorder=3)

    axes[1, 0].set_title(f"C. Feature Profile: Cell {cid}, Cyc {cyc} (Mode: {mode})", fontweight='bold', pad=15)
    axes[1, 0].set_xlabel("Feature Index")
    axes[1, 0].set_ylabel("Scaled HI Value")
    axes[1, 0].legend(frameon=True, shadow=True, title="Groups")
    axes[1, 0].grid(True, linestyle='--', alpha=0.4)
    axes[1, 0].set_xticks(range(0, 46, 5))
    sns.despine(ax=axes[1, 0])

    # --- [Plot 4: Trajectory of Top Feature over Cycles] ---
    best_feat = corrs_cap.idxmax()
    target_cell = df['cell'].iloc[0]
    df_cell = df[df['cell'] == target_cell].sort_values("cyc")

    color_feat, color_cap = '#8c564b', '#7f7f7f'
    sns.lineplot(data=df_cell, x="cyc", y=best_feat,
                 ax=axes[1, 1], color=color_feat, linewidth=2.5,
                 label=f"Feature ({best_feat})")

    axes[1, 1].set_title(
        f"D. Trajectory of Top Feature ({best_feat}) Over Cycles (Cell: {target_cell})",
        fontweight='bold',
        pad=15
    )
    axes[1, 1].set_xlabel("Cycle Number")
    axes[1, 1].set_ylabel(f"Scaled {best_feat} Value", color=color_feat, fontweight='bold')
    axes[1, 1].tick_params(axis='y', labelcolor=color_feat)

    ax2 = axes[1, 1].twinx()
    sns.lineplot(data=df_cell, x="cyc", y="cap_soh",
                 ax=ax2, color=color_cap, linewidth=2,
                 linestyle='--', label="Capacity (Ah)")

    ax2.set_ylabel("Capacity (Ah)", color=color_cap, fontweight='bold')
    ax2.tick_params(axis='y', labelcolor=color_cap)
    ax2.grid(False)

    lines_1, labels_1 = axes[1, 1].get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    axes[1, 1].legend(lines_1 + lines_2, labels_1 + labels_2,
                      loc='upper right', frameon=True, shadow=True)

    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Comprehensive HI Analysis saved to: {save_path}")


# 시각화 실행
if 'batch_feature_pool' in locals() and len(batch_feature_pool) > 0:
    plot_comprehensive_hi_analysis(batch_feature_pool)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

def deep_analyze_high_scenario(feature_pool, target_soc=-2):
    """
    특정 SOC 시나리오(기본값: High, -2)에 대해 집중 분석을 수행합니다.
    """
    # 1. 특정 시나리오 데이터 필터링
    high_samples = [s for s in feature_pool if s["soc_label"] == target_soc]
    target_soc_str = { -2: "High SOC", -1: "Mid SOC", 0: "Low SOC" }.get(target_soc, f"SOC {target_soc}")

    # 데이터 형태 샘플로 확인
    data_shape = (len(high_samples), len(high_samples[0]["scaled_x"])) if high_samples else (0, 0)
    print(f"Filtered {len(high_samples)} samples for soc_label {target_soc} with scaled_x shape: {data_shape}")
    
    if not high_samples:
        print(f"No samples found for soc_label {target_soc}")
        return

    # 데이터프레임 변환 (HI 45개 + 메타 정보)
    df = pd.DataFrame([{
        "cell": s["cell"], "cyc": s["cyc"], "mode": s["mode_label"],
        "cap": s["capacity"], "len": s.get("length_p", 0),
        **{f"f{i}": val for i, val in enumerate(s["scaled_x"][:45])}  # 순수 HI 45개만 분석
    } for s in high_samples])

    print(f"=== [Deep Analysis] High SOC Scenario (soc_label: {target_soc}) ===")
    print(f"1. Total Segments: {len(df)}")
    print(f"2. Unique Cells  : {df['cell'].nunique()}")
    print(f"3. Avg. Capacity : {df['cap'].mean():.4f} Ah")
    print("-" * 50)

    # 2. HI 통계치 분석
    hi_cols = [f"f{i}" for i in range(45)]
    hi_stats = df[hi_cols].describe().transpose()[['mean', 'std', 'min', 'max']]

    # 3. HI와 Capacity(SOH) 간의 상관관계 계산 (Spearman 사용 - 비선형 고려)
    correlations = []
    for col in hi_cols:
        corr, _ = spearmanr(df[col], df['cap'])
        correlations.append(corr)

    df_hi_analysis = hi_stats.copy()
    df_hi_analysis['SOH_Correlation'] = correlations
    df_hi_analysis['Abs_Corr'] = np.abs(correlations)

    # 상관관계 상위 10개 피처 출력
    top_10 = df_hi_analysis.sort_values('Abs_Corr', ascending=False).head(10)
    print("\n[Top 10 Health Indicators by Correlation with SOH]")
    print(top_10[['SOH_Correlation', 'mean', 'std']])

    # 4. 시각화 (3-Panel Plot)
    fig = plt.subplots(figsize=(22, 18))

    # (1) Correlation Bar Chart
    plt.subplot(3, 1, 1)
    colors = ['red' if c > 0 else 'blue' for c in correlations]
    plt.bar(range(45), correlations, color=colors, alpha=0.7, edgecolor='black')
    plt.axhline(0, color='black', lw=1)
    plt.title(f"Health Indicator Correlation with Capacity (Scenario: {target_soc_str})", fontsize=16, fontweight='bold')
    plt.xlabel("HI Index (f0-f44)")
    plt.ylabel("Spearman Correlation")
    plt.ylim(-0.3, 0.3)
    plt.xticks(range(0, 45, 1))
    plt.grid(True, axis='y', linestyle='--', alpha=0.6)

    # (2) Top 5 HI Trajectories (Sample Cell)
    plt.subplot(3, 2, 3)
    target_cell = df['cell'].iloc[0]
    df_sample = df[df['cell'] == target_cell].sort_values('cyc')
    top_5_feats = top_10.index[:5]

    for feat in top_5_feats:
        plt.plot(df_sample['cyc'], df_sample[feat], label=feat, lw=2)
    plt.title(f"Top 5 HI Trends (Cell: {target_cell})", fontsize=14)
    plt.xlabel("Cycle")
    plt.ylabel("Scaled Value")
    plt.legend()

    # (3) SOH vs Top Correlation Feature (Scatter)
    plt.subplot(3, 2, 4)
    best_feat = top_10.index[0]
    sns.regplot(data=df, x=best_feat, y='cap',
                scatter_kws={'alpha': 0.2, 'color': 'grey'},
                line_kws={'color': 'red'})
    plt.title(f"SOH vs. {best_feat} (Best Feature)", fontsize=14)
    plt.xlabel(f"Scaled {best_feat}")
    plt.ylabel("Capacity (Ah)")

    # (4) Distribution of Features (Heatmap)
    plt.subplot(3, 1, 3)
    sns.heatmap(df[hi_cols].iloc[::50].transpose(), cmap='RdBu_r', center=0)
    plt.title("Feature Value Intensity Map (Subset of Samples)", fontsize=14)
    plt.xlabel("Sample Index (Every 50th)")
    plt.ylabel("HI Index")

    plt.tight_layout()
    plt.savefig(f"{figure_path}/{DATASET_TYPE}_{target_soc_str}_soc_deep_analysis.png", dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()


    return df_hi_analysis


# 분석 실행
if 'batch_feature_pool' in locals() and len(batch_feature_pool) > 0:
    high_analysis_report = deep_analyze_high_scenario(batch_feature_pool, target_soc=0)
    high_analysis_report = deep_analyze_high_scenario(batch_feature_pool, target_soc=-1)
    high_analysis_report = deep_analyze_high_scenario(batch_feature_pool, target_soc=-2)

In [ ]:
import pickle
import numpy as np
from pathlib import Path

def inspect_optimized_tensors(data_root="D:/chanminLee/data_store/LFP_SOH_estimation"):
    """
    저장된 optimized_tensors.pkl 파일들을 읽어서 
    전체 샘플 수, 데이터 키(Key), 텐서 차원을 출력합니다.
    """
    root = Path(data_root)
    # 분석할 파일 목록 (mit, hust 등)
    target_files = list(root.glob("*_optimized_tensors.pkl"))
    
    if not target_files:
        print(f"[Error] No files found in {data_root}")
        return

    for file_path in target_files:
        print(f"\n{'='*60}")
        print(f" FILE: {file_path.name}")
        print(f"{'='*60}")

        try:
            with open(file_path, "rb") as f:
                data_pool = pickle.load(f)

            num_samples = len(data_pool)
            print(f"1. Total Sample Count: {num_samples:,}")

            if num_samples > 0:
                sample = data_pool[0]
                print(f"2. Data Keys: {list(sample.keys())}")

                print(f"3. Component Shapes & Types:")
                # 입력 피처(x) 확인
                if 'x' in sample:
                    x_val = sample['x']
                    print(f"   - [x] Input Features : {x_val.shape} (Dtype: {x_val.dtype})")

                    # 48차원 여부 체크 (45 HI + 3 Meta)
                    if x_val.shape[0] == 48:
                        print("     >> Status: 48D (Scenario A Meta-Injected) - CORRECT")
                    else:
                        print(f"     >> Status: {x_val.shape[0]}D - Needs reprocessing for 48D")

                # 타겟 값 확인
                if 'y' in sample:
                    print(f"   - [y] RUL Label      : Scalar ({type(sample['y']).__name__})")
                if 'capacity' in sample:
                    print(f"   - [capacity] SOH Label: Scalar ({type(sample['capacity']).__name__})")

                # 메타 데이터 예시 출력
                print(f"4. Sample Value Preview (Index 0):")
                print(f"   - Cell ID: {sample.get('cell')}, Cycle: {sample.get('cyc')}")
                print(f"   - Mode: {'Charge' if sample.get('mode_label')==1 else 'Discharge'}")
                if x_val.shape[0] >= 48:
                    print(f"   - Injected Meta (Last 3): {x_val[-3:]} (SOC, Mode, Length)")

        except Exception as e:
            print(f"[Error] Failed to read {file_path.name}: {e}")

# 점검 실행
inspect_optimized_tensors()

## Phase 6: Comprehensive Pipeline Diagnostics
Consolidated analysis of optimized tensors and intermediate sliced segments.

In [ ]:
def run_comprehensive_analysis(dataset_type=DATASET_TYPE, data_root=PROCESSED_DATA_ROOT):
    """
    [Unified Analysis] Tensor Inspection + Segment Statistics (Fixed Visualization)
    """
    import pickle
    from pathlib import Path
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from tqdm.auto import tqdm
    from datetime import date
    
    report_lines = []
    report_lines.append("="*80)
    report_lines.append(f" COMPREHENSIVE DATA ANALYSIS REPORT: {dataset_type.upper()}")
    report_lines.append(f" Date: {date.today().strftime('%Y-%m-%d')}")
    report_lines.append("="*80 + "\n")

    # --- Part A: Tensor Inspection ---
    tensor_path = Path(data_root) / f"{dataset_type}_optimized_tensors.pkl"
    report_lines.append("[Part A] Optimized Tensor Inspection")
    if not tensor_path.exists():
        report_lines.append(f"[Error] Tensor file not found: {tensor_path}")
    else:
        with open(tensor_path, "rb") as f: data_pool = pickle.load(f)
        num_samples = len(data_pool)
        report_lines.append(f"1. Total Sample Count: {num_samples:,}")
        if num_samples > 0:
            sample = data_pool[0]
            report_lines.append(f"2. Data Keys: {list(sample.keys())}")
            if 'x' in sample:
                x_shape = sample['x'].shape
                status = "CORRECT (48D)" if x_shape[0] == 48 else f"INCORRECT ({x_shape[0]}D)"
                report_lines.append(f"3. Feature Shape: {x_shape} -> {status}")
            report_lines.append(f"4. Sample Metadata: Cell={sample.get('cell')}, Cyc={sample.get('cyc')}, Mode={'Charge' if sample.get('mode_label')==1 else 'Discharge'}")
        del data_pool
    report_lines.append("\n")

    # --- Part B: Sliced Chunks Statistics ---
    cache_dir = Path(data_root) / f"{dataset_type}_sliced_chunks"
    batch_files = sorted(cache_dir.glob("batch_*.pkl"))
    report_lines.append("[Part B] Sliced Segment Statistics")
    if not batch_files:
        report_lines.append(f"[Error] No sliced batch files found in {cache_dir}")
    else:
        stats_list = []
        for bf in tqdm(batch_files, desc="Analyzing Batches"):
            with open(bf, "rb") as f: batch_data = pickle.load(f)
            segments = batch_data.get('segments', [])
            for item in segments:
                df = item.get('df')
                if df is None or df.empty: continue
                # Handle variable column names (t vs Time (s), V vs Voltage (V))
                t_col = 't' if 't' in df.columns else 'Time (s)'
                v_col = 'V' if 'V' in df.columns else 'Voltage (V)'
                
                stats_list.append({
                    'SOC_Scenario': item.get('soc_label'),
                    'Mode': item.get('mode_label'),
                    'Window_Size': item.get('length_p'),
                    'Row_Count': len(df),
                    'Time_Span_s': (df[t_col].max() - df[t_col].min()) if t_col in df.columns else np.nan,
                    'Voltage_Span_V': (df[v_col].max() - df[v_col].min()) if v_col in df.columns else np.nan
                })
        df_stats = pd.DataFrame(stats_list)
        df_stats['SOC_Scenario_Name'] = df_stats['SOC_Scenario'].map({-2: 'High', -1: 'Mid', 0: 'Low'})
        df_stats['Mode_Name'] = df_stats['Mode'].map({1: 'Charge', 0: 'Discharge'})
        
        grouped = df_stats.groupby(['Mode_Name', 'SOC_Scenario_Name']).agg(
            Count=('Row_Count', 'count'), Row_Mean=('Row_Count', 'mean'), Time_Mean_s=('Time_Span_s', 'mean'), Volt_Mean_V=('Voltage_Span_V', 'mean')
        ).reset_index()
        report_lines.append(grouped.to_string(index=False))
        
        # --- Visualization (Simplified to avoid boxprops error) ---
        fig, axes = plt.subplots(2, 2, figsize=(20, 16))
        
        # 1. Barplot
        sns.barplot(data=grouped, x='SOC_Scenario_Name', y='Count', hue='Mode_Name', ax=axes[0,0])
        axes[0,0].set_title("Segment Count by Scenario & Mode")
        
        # 2. Row Count Boxplot
        sns.boxplot(data=df_stats, x='SOC_Scenario_Name', y='Row_Count', hue='Mode_Name', ax=axes[0,1])
        axes[0,1].set_title("Row Count Distribution")
        
        # 3. Time Span Boxplot
        sns.boxplot(data=df_stats, x='SOC_Scenario_Name', y='Time_Span_s', hue='Mode_Name', ax=axes[1,0])
        axes[1,0].set_title("Time Span (s) Distribution")
        
        # 4. Voltage Span Violinplot
        sns.violinplot(data=df_stats, x='SOC_Scenario_Name', y='Voltage_Span_V', hue='Mode_Name', split=True, ax=axes[1,1])
        axes[1,1].set_title("Voltage Span (V) Distribution")
        
        plt.tight_layout()
        vis_path = FIGURE_ROOT / f"{dataset_type}_comprehensive_analysis.png"
        plt.savefig(vis_path, dpi=300, bbox_inches='tight')
        plt.show()
        report_lines.append(f"\n[Success] Visualization saved to: {vis_path}")

    # --- Save Report ---
    report_dir = PROJECT_ROOT / "outputs" / "reports"
    report_dir.mkdir(parents=True, exist_ok=True)
    report_path = report_dir / f"{dataset_type}_comprehensive_analysis.txt"
    with open(report_path, "w", encoding="utf-8") as f: f.write("\n".join(report_lines))
    print(f"\n[Success] Comprehensive report saved to: {report_path}")
    print("\n".join(report_lines))

run_comprehensive_analysis()